In [2]:
import pandas as pd

path = "/storage/vacoda/data/interim/hs_sync/target_h5/applied/comtrade_target_h5_hs_synced.parquet"

df = pd.read_parquet(path)

In [15]:
df.head(3)

,hs_sync_input_row_id,type_code,frequency_code,ref_period_id,year,month,period,reporter_code,reporter_iso,reporter_name,...,source_cmd_original,target_hs,target_cmd,allocation_weight,conversion_need,hs_sync_method,allocation_strategy,allocation_is_assumption,primary_value_original,primary_value_hs_synced
0,1,C,A,20170101,2017,52,2017,28,ATG,Antigua and Barbuda,...,391211,H5,391211,1.0,easy_without_weights,direct_1_to_1,deterministic,False,169.270,169.270
1,2,C,A,20170101,2017,52,2017,28,ATG,Antigua and Barbuda,...,391211,H5,391211,1.0,easy_without_weights,direct_1_to_1,deterministic,False,169.270,169.270
2,3,C,A,20170101,2017,52,2017,28,ATG,Antigua and Barbuda,...,391211,H5,391211,1.0,easy_without_weights,direct_1_to_1,deterministic,False,38.952,38.952


In [4]:
df.columns.tolist()

['hs_sync_input_row_id',
 'type_code',
 'frequency_code',
 'ref_period_id',
 'year',
 'month',
 'period',
 'reporter_code',
 'reporter_iso',
 'reporter_name',
 'flow_code',
 'flow_name',
 'partner_code',
 'partner_iso',
 'partner_name',
 'partner2_code',
 'partner2_iso',
 'partner2_name',
 'source_hs',
 'classification_search',
 'is_original_classification',
 'source_cmd',
 'commodity_name',
 'aggregation_level',
 'is_leaf',
 'customs_code',
 'customs_name',
 'mos_code',
 'mot_code',
 'mot_name',
 'quantity_unit_code',
 'quantity_unit',
 'quantity',
 'is_quantity_estimated',
 'alt_quantity_unit_code',
 'alt_quantity_unit',
 'alt_quantity',
 'is_alt_quantity_estimated',
 'net_weight_kg',
 'is_net_weight_estimated',
 'gross_weight_kg',
 'is_gross_weight_estimated',
 'cif_value_usd',
 'fob_value_usd',
 'primary_value',
 'legacy_estimation_flag',
 'is_reported',
 'is_aggregate',
 'source_hs_original',
 'source_cmd_original',
 'target_hs',
 'target_cmd',
 'allocation_weight',
 'conversion_n

In [6]:
value_columns =['cif_value_usd',
 'fob_value_usd',
 'primary_value']

df[value_columns].describe()

,cif_value_usd,fob_value_usd,primary_value
count,3.345576e+07,3.344003e+07,4.991792e+07
mean,8.056602e+05,1.088909e+06,1.187314e+06
std,1.709283e+07,2.401075e+07,2.327669e+07
min,0.000000e+00,0.000000e+00,0.000000e+00
25%,7.119700e+01,1.159340e+02,4.822450e+02
50%,1.992000e+03,3.340467e+03,6.148100e+03
75%,3.630400e+04,5.159355e+04,7.244236e+04
max,1.139287e+10,1.298835e+10,1.298835e+10


In [7]:
value_cols = [
    "primary_value",
    "cif_value_usd",
    "fob_value_usd",
]

df[value_cols].isna().value_counts().sort_index()

primary_value  cif_value_usd  fob_value_usd
False          False          False            16977876
                              True             16477885
               True           False            16462156
Name: count, dtype: int64

In [8]:
pd.DataFrame({
    "rows": [
        len(df),
        df["primary_value"].isna().sum(),
        (
            df["primary_value"].isna()
            & df["cif_value_usd"].notna()
        ).sum(),
        (
            df["primary_value"].isna()
            & df["fob_value_usd"].notna()
        ).sum(),
        (
            df["primary_value"].isna()
            & (
                df["cif_value_usd"].notna()
                | df["fob_value_usd"].notna()
            )
        ).sum(),
    ]
}, index=[
    "total_rows",
    "primary_missing",
    "primary_missing_but_cif_present",
    "primary_missing_but_fob_present",
    "primary_missing_but_any_alternative_present",
])

,rows
total_rows,49917917
primary_missing,0
primary_missing_but_cif_present,0
primary_missing_but_fob_present,0
primary_missing_but_any_alternative_present,0


In [9]:
df[value_cols].eq(0).sum()

primary_value         44
cif_value_usd    5202168
fob_value_usd    4077697
dtype: int64

In [10]:
availability = (
    df[
        [
            "primary_value",
            "cif_value_usd",
            "fob_value_usd",
        ]
    ]
    .notna()
    .value_counts()
    .rename("rows")
    .reset_index()
)

availability

,primary_value,cif_value_usd,fob_value_usd,rows
0,True,True,True,16977876
1,True,True,False,16477885
2,True,False,True,16462156


In [11]:
comparison = pd.DataFrame({
    "primary_equals_cif": (
        df["primary_value"].eq(df["cif_value_usd"])
        & df["cif_value_usd"].notna()
    ),
    "primary_equals_fob": (
        df["primary_value"].eq(df["fob_value_usd"])
        & df["fob_value_usd"].notna()
    ),
})

comparison.value_counts().rename("rows")

primary_equals_cif  primary_equals_fob
True                False                 26510456
False               True                  21732470
True                True                   1674991
Name: rows, dtype: int64

In [12]:
pd.Series({
    "primary equals CIF": (
        df["primary_value"].eq(df["cif_value_usd"])
        & df["cif_value_usd"].notna()
    ).sum(),

    "primary equals FOB": (
        df["primary_value"].eq(df["fob_value_usd"])
        & df["fob_value_usd"].notna()
    ).sum(),

    "primary equals both": (
        df["primary_value"].eq(df["cif_value_usd"])
        & df["primary_value"].eq(df["fob_value_usd"])
        & df["cif_value_usd"].notna()
        & df["fob_value_usd"].notna()
    ).sum(),

    "primary differs from both": (
        df["cif_value_usd"].notna()
        & df["fob_value_usd"].notna()
        & df["primary_value"].ne(df["cif_value_usd"])
        & df["primary_value"].ne(df["fob_value_usd"])
    ).sum(),
})


primary equals CIF           28185447
primary equals FOB           23407461
primary equals both           1674991
primary differs from both           0
dtype: int64

In [13]:
df.assign(
    primary_equals_cif=(
        df["primary_value"].eq(df["cif_value_usd"])
        & df["cif_value_usd"].notna()
    ),
    primary_equals_fob=(
        df["primary_value"].eq(df["fob_value_usd"])
        & df["fob_value_usd"].notna()
    ),
).groupby("flow_code").agg(
    rows=("primary_value", "size"),
    cif_available=("cif_value_usd", "count"),
    fob_available=("fob_value_usd", "count"),
    primary_equals_cif=("primary_equals_cif", "sum"),
    primary_equals_fob=("primary_equals_fob", "sum"),
)

,rows,cif_available,fob_available,primary_equals_cif,primary_equals_fob
flow_code,,,,,
DX,4485514,1100496,4485514,3478,4485514
FM,4465801,4275553,1815395,4095596,563843
M,24938179,23914484,11396053,23573211,2766223
MIP,126993,126993,71503,126993,6108
MOP,28759,28759,8960,28759,440
RM,373089,341063,163033,333977,85759
RX,409186,156055,409186,3924,409186
X,14952413,3442777,14952405,19297,14952405
XIP,34433,9468,34433,0,34433


In [16]:
keys = [
    "period",
    "reporter_code",
    "partner_code",
    "target_cmd",
]

export_check = (
    df[df["flow_code"].isin(["X", "DX", "RX"])]
    .groupby(keys + ["flow_code"], as_index=False)["primary_value"]
    .sum()
    .pivot_table(
        index=keys,
        columns="flow_code",
        values="primary_value",
        fill_value=0,
    )
    .reset_index()
)

export_check["dx_plus_rx"] = (
    export_check.get("DX", 0)
    + export_check.get("RX", 0)
)

export_check["difference"] = (
    export_check.get("X", 0)
    - export_check["dx_plus_rx"]
)

export_check["relative_difference"] = (
    export_check["difference"].abs()
    / export_check.get("X", 0).replace(0, pd.NA)
)

export_check["relative_difference"].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
)

count     3867229.0
unique      49075.0
top             1.0
freq      2859467.0
Name: relative_difference, dtype: float64

In [17]:
pd.Series({
    "comparable_groups": export_check["relative_difference"].notna().sum(),
    "within_1_percent": export_check["relative_difference"].le(0.01).sum(),
    "within_5_percent": export_check["relative_difference"].le(0.05).sum(),
    "within_10_percent": export_check["relative_difference"].le(0.10).sum(),
})

comparable_groups    3867229
within_1_percent     1002841
within_5_percent     1003356
within_10_percent    1003671
dtype: int64

In [18]:
import numpy as np

x_value = export_check.get("X", 0)

export_check["relative_difference"] = np.where(
    x_value > 0,
    export_check["difference"].abs() / x_value,
    np.nan,
)

export_check["relative_difference"].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
)

count    3.867229e+06
mean     7.401726e-01
std      4.383936e-01
min      0.000000e+00
50%      1.000000e+00
75%      1.000000e+00
90%      1.000000e+00
95%      1.000000e+00
99%      1.000000e+00
max      1.000000e+00
Name: relative_difference, dtype: float64

In [19]:

comparable = export_check.loc[
    export_check.get("X", 0).gt(0)
    & (
        export_check.get("DX", 0).gt(0)
        | export_check.get("RX", 0).gt(0)
    )
].copy()

comparable["dx_plus_rx"] = (
    comparable.get("DX", 0)
    + comparable.get("RX", 0)
)

comparable["relative_difference"] = (
    (
        comparable["X"]
        - comparable["dx_plus_rx"]
    ).abs()
    / comparable["X"]
)

pd.Series({
    "comparable_groups": len(comparable),
    "within_1_percent": comparable["relative_difference"].le(0.01).sum(),
    "within_5_percent": comparable["relative_difference"].le(0.05).sum(),
    "within_10_percent": comparable["relative_difference"].le(0.10).sum(),
})

comparable_groups    1007762
within_1_percent     1002841
within_5_percent     1003356
within_10_percent    1003671
dtype: int64

In [20]:
pd.Series({
    "within_1_percent_pct":
        comparable["relative_difference"].le(0.01).mean() * 100,

    "within_5_percent_pct":
        comparable["relative_difference"].le(0.05).mean() * 100,

    "within_10_percent_pct":
        comparable["relative_difference"].le(0.10).mean() * 100,
})

within_1_percent_pct     99.511690
within_5_percent_pct     99.562794
within_10_percent_pct    99.594051
dtype: float64

In [21]:
export_check["has_x"] = export_check.get("X", 0).gt(0)
export_check["has_dx"] = export_check.get("DX", 0).gt(0)
export_check["has_rx"] = export_check.get("RX", 0).gt(0)

export_check["has_detailed_export"] = (
    export_check["has_dx"]
    | export_check["has_rx"]
)

pd.Series({
    "groups_with_x": export_check["has_x"].sum(),

    "groups_with_detailed_export":
        export_check["has_detailed_export"].sum(),

    "detailed_groups_without_x": (
        ~export_check["has_x"]
        & export_check["has_detailed_export"]
    ).sum(),

    "dx_without_x": (
        ~export_check["has_x"]
        & export_check["has_dx"]
    ).sum(),

    "rx_without_x": (
        ~export_check["has_x"]
        & export_check["has_rx"]
    ).sum(),
})

groups_with_x                  3867229
groups_with_detailed_export    1007762
detailed_groups_without_x            0
dx_without_x                         0
rx_without_x                         0
dtype: int64

In [22]:
export_fallback = export_check.loc[
    ~export_check["has_x"]
    & export_check["has_detailed_export"]
].copy()

pd.Series({
    "fallback_dx_value":
        export_fallback.get("DX", 0).sum(),

    "fallback_rx_value":
        export_fallback.get("RX", 0).sum(),

    "fallback_total_export_value":
        (
            export_fallback.get("DX", 0)
            + export_fallback.get("RX", 0)
        ).sum(),
})

fallback_dx_value              0.0
fallback_rx_value              0.0
fallback_total_export_value    0.0
dtype: float64

In [24]:
## 
value_columns = ['primary_value', 'primary_value_hs_synced', 'primary_value_original']
df[value_columns].describe()

,primary_value,primary_value_hs_synced,primary_value_original
count,4.991792e+07,4.991792e+07,4.991792e+07
mean,1.187314e+06,1.163947e+06,1.187314e+06
std,2.327669e+07,2.191921e+07,2.327669e+07
min,0.000000e+00,0.000000e+00,0.000000e+00
25%,4.822450e+02,4.736500e+02,4.822450e+02
50%,6.148100e+03,6.033479e+03,6.148100e+03
75%,7.244236e+04,7.107164e+04,7.244236e+04
max,1.298835e+10,1.298835e+10,1.298835e+10


In [25]:
import duckdb

path = (
    "/storage/vacoda/data/interim/cif_fob/"
    "target_h5/target_h5_cif_fob_mirror_pairs.parquet"
)

df = duckdb.sql(
    f"""
    SELECT *
    FROM read_parquet('{path}')
    LIMIT 100
    """
).df()

df.head()

,year,exporter_iso,importer_iso,target_cmd,export_value_fob,import_value_cif,export_net_weight_kg,import_net_weight_kg,export_direct_fob_value,export_direct_cif_value,...,has_export_report,has_import_report,has_both_mirrors,raw_value_ratio,export_unit_value_usd_per_kg,import_unit_value_usd_per_kg,unit_value_ratio,quantity_concordance,exclusion_reason,training_eligible
0,2023,USA,NLD,440399,1.237298e+06,4.291339e+04,4.608168e+06,27263.0,1.237298e+06,NaN,...,True,True,True,0.034683,0.268501,1.574052,5.862370,0.005916,low_quantity_concordance,False
1,2023,USA,THA,440399,6.409000e+04,6.794644e+03,2.386955e+05,8834.0,6.409000e+04,NaN,...,True,True,True,0.106017,0.268501,0.769147,2.864596,0.037009,low_quantity_concordance,False
2,2024,BEL,PRT,440399,2.460930e+02,4.825988e+04,7.352000e+01,69513.0,2.460930e+02,NaN,...,True,True,True,196.104249,3.347293,0.694257,0.207408,0.001058,low_quantity_concordance,False
3,2024,CAN,CHN,440399,1.065437e+07,1.403722e+07,4.277231e+07,22764763.0,1.065437e+07,NaN,...,True,True,True,1.317507,0.249095,0.616620,2.475441,0.532231,low_quantity_concordance,False
4,2024,HRV,BIH,440399,8.581400e+04,1.365331e+05,4.841800e+05,1090633.0,8.581400e+04,NaN,...,True,True,True,1.591035,0.177236,0.125187,0.706330,0.443944,low_quantity_concordance,False


In [26]:
duckdb.sql(
    f"""
    SELECT
        exclusion_reason,
        COUNT(*) AS n_rows,
        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            2
        ) AS share_percent
    FROM read_parquet('{path}')
    GROUP BY exclusion_reason
    ORDER BY n_rows DESC
    """
).df()

,exclusion_reason,n_rows,share_percent
0,low_quantity_concordance,1972869,35.95
1,missing_export_report,1831489,33.37
2,missing_import_report,1328738,24.21
3,eligible,303681,5.53
4,missing_or_nonpositive_import_weight,29993,0.55
5,missing_or_nonpositive_export_weight,21210,0.39
6,nonpositive_export_value,1,0.00


In [27]:
duckdb.sql(
    f"""
    SELECT
        QUANTILE_CONT(
            quantity_concordance,
            [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
        ) AS concordance_quantiles
    FROM read_parquet('{path}')
    WHERE quantity_concordance IS NOT NULL
    """
).df()

,concordance_quantiles
0,"[0.0001696709527056308, 0.0034566062186056684,..."


In [29]:
duckdb.sql(
    f"""
    SELECT
        COUNT(*) AS n_eligible,
        MEDIAN(raw_value_ratio) AS median_value_ratio,
        MEDIAN(unit_value_ratio) AS median_unit_value_ratio,
        QUANTILE_CONT(
            unit_value_ratio,
            [0.01, 0.05, 0.50, 0.95, 0.99]
        ) AS unit_value_ratio_quantiles
    FROM read_parquet('{path}')
    WHERE training_eligible
    """
).df()

,n_eligible,median_value_ratio,median_unit_value_ratio,unit_value_ratio_quantiles
0,303681,1.047962,1.048389,"[0.1622232548794118, 0.5279949415938775, 1.048..."


In [30]:
import duckdb

path = (
    "/storage/vacoda/data/interim/cif_fob/"
    "target_h5/target_h5_cif_fob_mirror_pairs.parquet"
)

duckdb.sql(
    f"""
    SELECT
        COUNT(*) AS n_eligible,

        COUNT(*) FILTER (
            WHERE unit_value_ratio < 1
        ) AS n_ratio_below_1,

        COUNT(*) FILTER (
            WHERE unit_value_ratio BETWEEN 1 AND 1.5
        ) AS n_ratio_1_to_1_5,

        COUNT(*) FILTER (
            WHERE unit_value_ratio > 1.5
        ) AS n_ratio_above_1_5,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE unit_value_ratio < 1
            ) / COUNT(*),
            2
        ) AS share_below_1_percent

    FROM read_parquet('{path}')
    WHERE training_eligible
    """
).df()

,n_eligible,n_ratio_below_1,n_ratio_1_to_1_5,n_ratio_above_1_5,share_below_1_percent
0,303681,97331,174653,31697,32.05


In [33]:
import duckdb

path_hs_sync = (
    "/storage/vacoda/data/interim/hs_sync/target_h5/applied/"
    "comtrade_target_h5_hs_synced.parquet"
)

duckdb.sql(
    f"""
    SELECT
        is_quantity_estimated,
        is_net_weight_estimated,
        COUNT(*) AS n_rows,

        MEDIAN(quantity / net_weight_kg) FILTER (
            WHERE quantity > 0
              AND net_weight_kg > 0
              AND LOWER(quantity_unit) = 'kg'
        ) AS median_quantity_to_net_weight

    FROM read_parquet('{path_hs_sync}')

    GROUP BY
        is_quantity_estimated,
        is_net_weight_estimated

    ORDER BY
        is_quantity_estimated,
        is_net_weight_estimated
    """
).df()

,is_quantity_estimated,is_net_weight_estimated,n_rows,median_quantity_to_net_weight
0,False,False,41037308,1.000000
1,False,True,557736,0.999173
2,True,False,2800718,1.161457
3,True,True,5522155,1.000000


In [34]:
duckdb.sql(
    f"""
    SELECT
        is_quantity_estimated,
        is_net_weight_estimated,

        COUNT(*) FILTER (
            WHERE quantity > 0
              AND net_weight_kg > 0
              AND LOWER(TRIM(quantity_unit)) = 'kg'
        ) AS n_comparable_kg_rows,

        MEDIAN(quantity / net_weight_kg) FILTER (
            WHERE quantity > 0
              AND net_weight_kg > 0
              AND LOWER(TRIM(quantity_unit)) = 'kg'
        ) AS median_quantity_to_net_weight,

        QUANTILE_CONT(
            quantity / net_weight_kg,
            [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
        ) FILTER (
            WHERE quantity > 0
              AND net_weight_kg > 0
              AND LOWER(TRIM(quantity_unit)) = 'kg'
        ) AS ratio_quantiles

    FROM read_parquet('{path_hs_sync}')

    GROUP BY
        is_quantity_estimated,
        is_net_weight_estimated

    ORDER BY
        is_quantity_estimated,
        is_net_weight_estimated
    """
).df()

,is_quantity_estimated,is_net_weight_estimated,n_comparable_kg_rows,median_quantity_to_net_weight,ratio_quantiles
0,False,False,37547512,1.000000,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]"
1,False,True,2,0.999173,"[0.9991730843553875, 0.9991730843553874, 0.999..."
2,True,False,570,1.161457,"[0.011277777777777779, 0.3178672421601963, 0.7..."
3,True,True,5035355,1.000000,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]"


In [35]:
import duckdb

path_hs_sync = (
    "/storage/vacoda/data/interim/hs_sync/target_h5/applied/"
    "comtrade_target_h5_hs_synced.parquet"
)

duckdb.sql(
    f"""
    WITH classified AS (
        SELECT
            *,

            net_weight_kg > 0
                AS has_net_weight,

            quantity > 0
            AND LOWER(TRIM(quantity_unit)) = 'kg'
                AS has_quantity_kg,

            alt_quantity > 0
            AND LOWER(TRIM(alt_quantity_unit)) = 'kg'
                AS has_alt_quantity_kg

        FROM read_parquet('{path_hs_sync}')
        WHERE flow_code IN ('M', 'X')
    )

    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE has_net_weight
        ) AS rows_with_net_weight,

        COUNT(*) FILTER (
            WHERE NOT has_net_weight
              AND has_quantity_kg
        ) AS recovered_by_quantity_kg,

        COUNT(*) FILTER (
            WHERE NOT has_net_weight
              AND NOT has_quantity_kg
              AND has_alt_quantity_kg
        ) AS recovered_by_alt_quantity_kg,

        COUNT(*) FILTER (
            WHERE NOT has_net_weight
              AND NOT has_quantity_kg
              AND NOT has_alt_quantity_kg
        ) AS still_without_kg_measure,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE NOT has_net_weight
                  AND NOT has_quantity_kg
                  AND has_alt_quantity_kg
            ) / COUNT(*),
            4
        ) AS alt_quantity_recovery_share_percent

    FROM classified
    """
).df()

,total_rows,rows_with_net_weight,recovered_by_quantity_kg,recovered_by_alt_quantity_kg,still_without_kg_measure,alt_quantity_recovery_share_percent
0,39890592,39437037,61075,19315,53234,0.0484


In [36]:
duckdb.sql(
    f"""
    WITH classified AS (
        SELECT
            *,

            net_weight_kg > 0
                AS has_net_weight,

            quantity > 0
            AND LOWER(TRIM(quantity_unit)) = 'kg'
                AS has_quantity_kg,

            alt_quantity > 0
            AND LOWER(TRIM(alt_quantity_unit)) = 'kg'
                AS has_alt_quantity_kg,

            gross_weight_kg > 0
                AS has_gross_weight

        FROM read_parquet('{path_hs_sync}')
        WHERE flow_code IN ('M', 'X')
    )

    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE has_net_weight
        ) AS rows_with_net_weight,

        COUNT(*) FILTER (
            WHERE NOT has_net_weight
              AND has_quantity_kg
        ) AS recovered_by_quantity_kg,

        COUNT(*) FILTER (
            WHERE NOT has_net_weight
              AND NOT has_quantity_kg
              AND has_alt_quantity_kg
        ) AS recovered_by_alt_quantity_kg,

        COUNT(*) FILTER (
            WHERE NOT has_net_weight
              AND NOT has_quantity_kg
              AND NOT has_alt_quantity_kg
              AND has_gross_weight
        ) AS recovered_by_gross_weight,

        COUNT(*) FILTER (
            WHERE NOT has_net_weight
              AND NOT has_quantity_kg
              AND NOT has_alt_quantity_kg
              AND NOT has_gross_weight
        ) AS still_without_any_kg_measure,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE NOT has_net_weight
                  AND NOT has_quantity_kg
                  AND NOT has_alt_quantity_kg
                  AND has_gross_weight
            ) / COUNT(*),
            4
        ) AS gross_weight_recovery_share_percent

    FROM classified
    """
).df()

,total_rows,rows_with_net_weight,recovered_by_quantity_kg,recovered_by_alt_quantity_kg,recovered_by_gross_weight,still_without_any_kg_measure,gross_weight_recovery_share_percent
0,39890592,39437037,61075,19315,1646,51580,0.0041


In [37]:
duckdb.sql(
    f"""
    WITH candidates AS (
        SELECT
            gross_weight_kg,
            is_gross_weight_estimated
        FROM read_parquet('{path_hs_sync}')
        WHERE flow_code IN ('M', 'X')

          AND (
              net_weight_kg IS NULL
              OR net_weight_kg <= 0
          )

          AND NOT (
              quantity > 0
              AND LOWER(TRIM(quantity_unit)) = 'kg'
          )

          AND NOT (
              alt_quantity > 0
              AND LOWER(TRIM(alt_quantity_unit)) = 'kg'
          )

          AND gross_weight_kg > 0
    )

    SELECT
        COALESCE(is_gross_weight_estimated, FALSE)
            AS is_gross_weight_estimated,
        COUNT(*) AS n_rows,
        SUM(gross_weight_kg) AS total_gross_weight_kg

    FROM candidates

    GROUP BY
        COALESCE(is_gross_weight_estimated, FALSE)

    ORDER BY
        is_gross_weight_estimated
    """
).df()

,is_gross_weight_estimated,n_rows,total_gross_weight_kg
0,False,17221,3.827856e+10


In [38]:
duckdb.sql(
    f"""
    WITH classified AS (
        SELECT
            COALESCE(net_weight_kg > 0, FALSE)
                AS has_net_weight,

            COALESCE(
                quantity > 0
                AND LOWER(TRIM(quantity_unit)) = 'kg',
                FALSE
            ) AS has_quantity_kg,

            COALESCE(
                alt_quantity > 0
                AND LOWER(TRIM(alt_quantity_unit)) = 'kg',
                FALSE
            ) AS has_alt_quantity_kg,

            COALESCE(gross_weight_kg > 0, FALSE)
                AS has_gross_weight

        FROM read_parquet('{path_hs_sync}')
        WHERE flow_code IN ('M', 'X')
    )

    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE has_net_weight
        ) AS rows_with_net_weight,

        COUNT(*) FILTER (
            WHERE NOT has_net_weight
              AND has_quantity_kg
        ) AS recovered_by_quantity_kg,

        COUNT(*) FILTER (
            WHERE NOT has_net_weight
              AND NOT has_quantity_kg
              AND has_alt_quantity_kg
        ) AS recovered_by_alt_quantity_kg,

        COUNT(*) FILTER (
            WHERE NOT has_net_weight
              AND NOT has_quantity_kg
              AND NOT has_alt_quantity_kg
              AND has_gross_weight
        ) AS recovered_by_gross_weight,

        COUNT(*) FILTER (
            WHERE NOT has_net_weight
              AND NOT has_quantity_kg
              AND NOT has_alt_quantity_kg
              AND NOT has_gross_weight
        ) AS still_without_any_kg_measure

    FROM classified
    """
).df()

,total_rows,rows_with_net_weight,recovered_by_quantity_kg,recovered_by_alt_quantity_kg,recovered_by_gross_weight,still_without_any_kg_measure
0,39890592,39437037,121587,118203,17221,196544


In [1]:
import duckdb

training_path = (
    "/storage/vacoda/data/interim/cif_fob/target_h5/"
    "target_h5_cif_fob_training_sample.parquet"
)

duckdb.sql(
    f"""
    SELECT
        factor_source,
        COUNT(*) AS n_rows,

        MEDIAN(observed_cif_fob_factor) AS median_factor,

        QUANTILE_CONT(
            observed_cif_fob_factor,
            0.01
        ) AS p01,

        QUANTILE_CONT(
            observed_cif_fob_factor,
            0.05
        ) AS p05,

        QUANTILE_CONT(
            observed_cif_fob_factor,
            0.95
        ) AS p95,

        QUANTILE_CONT(
            observed_cif_fob_factor,
            0.99
        ) AS p99,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE observed_cif_fob_factor < 1
            ) / COUNT(*),
            2
        ) AS share_below_1_percent,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE observed_cif_fob_factor > 1.5
            ) / COUNT(*),
            2
        ) AS share_above_1_5_percent

    FROM read_parquet('{training_path}')

    WHERE model_eligible

    GROUP BY factor_source
    ORDER BY factor_source
    """
).df()

,factor_source,n_rows,median_factor,p01,p05,p95,p99,share_below_1_percent,share_above_1_5_percent
0,direct_import_cif_fob,683328,1.074113,1.000000,1.000000,1.735534,6.77770,0.57,7.14
1,mirror_unit_value,252947,1.044246,0.155277,0.516059,2.133497,7.46041,33.26,10.52


In [2]:
duckdb.sql(
    f"""
    SELECT
        COUNT(*) AS n_with_both_factors,

        MEDIAN(direct_cif_fob_factor)
            AS median_direct_factor,

        MEDIAN(mirror_unit_value_factor)
            AS median_mirror_factor,

        MEDIAN(
            direct_cif_fob_factor
            / mirror_unit_value_factor
        ) AS median_direct_to_mirror_ratio,

        QUANTILE_CONT(
            direct_cif_fob_factor
            / mirror_unit_value_factor,
            [0.05, 0.25, 0.50, 0.75, 0.95]
        ) AS comparison_quantiles

    FROM read_parquet('{training_path}')

    WHERE direct_cif_fob_factor > 0
      AND mirror_unit_value_factor > 0
      AND high_weight_concordance
    """
).df()

,n_with_both_factors,median_direct_factor,median_mirror_factor,median_direct_to_mirror_ratio,comparison_quantiles
0,51899,1.067795,1.067709,1.000222,"[0.5697458407026997, 0.9343470245644956, 1.000..."


In [3]:
duckdb.sql(
    f"""
    SELECT
        target_cmd,
        COUNT(*) AS n_rows,
        MEDIAN(direct_cif_fob_factor) AS median_factor,
        QUANTILE_CONT(direct_cif_fob_factor, 0.95) AS p95,
        QUANTILE_CONT(direct_cif_fob_factor, 0.99) AS p99
    FROM read_parquet('{training_path}')
    WHERE direct_cif_fob_factor > 0
    GROUP BY target_cmd
    HAVING COUNT(*) >= 100
    ORDER BY p99 DESC
    LIMIT 50
    """
).df()

,target_cmd,n_rows,median_factor,p95,p99
0,470329,835,1.074114,2.077611,56.402300
1,480210,1688,1.074156,2.817175,35.914677
2,470500,317,1.081913,1.889326,35.849162
3,470691,432,1.072987,1.571499,31.554233
4,470200,426,1.061625,1.431165,28.367001
5,482040,1946,1.101184,4.093559,27.790923
6,440121,776,1.066825,2.186657,25.269549
7,470620,708,1.094109,1.924714,24.282700
8,470100,556,1.090590,1.765897,23.791935
9,481099,2975,1.081860,2.420768,22.191711


## Inspect Mirror-Factor Cases

In [ ]:

import duckdb
import pandas as pd

training_path = (
    "/storage/vacoda/data/interim/cif_fob/target_h5/"
    "target_h5_cif_fob_training_sample.parquet"
)

mirror_examples = duckdb.sql(
    f"""
    SELECT
        year,
        exporter_iso,
        importer_iso,
        target_cmd,

        export_value_fob,
        import_value_cif,

        export_weight_kg,
        import_weight_kg,
        weight_concordance,

        export_unit_value_usd_per_kg,
        import_unit_value_usd_per_kg,

        raw_value_ratio,
        mirror_unit_value_factor,

        export_source_rows,
        import_source_rows,

        export_proxy_weight_rows,
        import_proxy_weight_rows,
        export_estimated_weight_rows,
        import_estimated_weight_rows

    FROM read_parquet('{training_path}')

    WHERE factor_source = 'mirror_unit_value'

    ORDER BY RANDOM()
    LIMIT 20
    """
).df()

mirror_examples

,year,exporter_iso,importer_iso,target_cmd,export_value_fob,import_value_cif,export_weight_kg,import_weight_kg,weight_concordance,export_unit_value_usd_per_kg,import_unit_value_usd_per_kg,raw_value_ratio,mirror_unit_value_factor,export_source_rows,import_source_rows,export_proxy_weight_rows,import_proxy_weight_rows,export_estimated_weight_rows,import_estimated_weight_rows
0,2023,CHN,GBR,491191,1.322924e+07,1.584340e+07,2.848463e+06,2.962048e+06,0.961653,4.644345,5.348801,1.197605,1.151680,1,31,0,0,0,1
1,2024,POL,BIH,480258,4.821900e+04,5.309173e+04,4.150000e+04,4.210356e+04,0.985665,1.161904,1.260980,1.101054,1.085270,1,4,0,0,0,0
2,2018,RUS,AZE,480257,1.332932e+06,1.338436e+06,1.461943e+06,1.446110e+06,0.989170,0.911754,0.925543,1.004129,1.015123,1,1,0,0,0,0
3,2019,TUR,HUN,940161,7.676310e+05,9.132710e+05,1.060120e+05,1.091920e+05,0.970877,7.240982,8.363900,1.189727,1.155078,2,1,0,0,0,0
4,2019,CHE,CRI,491000,4.678920e+02,7.371400e+02,3.900000e+01,4.090100e+01,0.953522,11.997231,18.022542,1.575449,1.502225,2,1,0,0,0,1
5,2019,BRA,IND,480100,8.545000e+04,1.052763e+05,2.034530e+05,2.030000e+05,0.997773,0.419999,0.518603,1.232023,1.234772,1,1,0,0,0,0
6,2022,SWE,SGP,480100,2.744840e+06,3.116558e+06,3.680000e+06,3.815650e+06,0.964449,0.745880,0.816783,1.135424,1.095059,1,1,0,0,0,0
7,2018,LBN,AUT,481910,4.200000e+01,1.709029e+03,2.504700e+01,2.500000e+01,0.998124,1.676848,68.361160,40.691167,40.767666,1,1,0,0,1,0
8,2019,LUX,DEU,481013,1.870544e+05,1.881825e+05,9.135319e+04,8.883400e+04,0.972424,2.047596,2.118362,1.006031,1.034560,2,2,0,0,0,0
9,2021,DEU,MMR,481200,2.055515e+04,2.351414e+04,1.836000e+03,1.971625e+03,0.931212,11.195614,11.926274,1.143954,1.065263,2,2,0,0,0,0


In [5]:
mirror_examples = mirror_examples.assign(
    calculated_export_unit_value=(
        mirror_examples["export_value_fob"]
        / mirror_examples["export_weight_kg"]
    ),
    calculated_import_unit_value=(
        mirror_examples["import_value_cif"]
        / mirror_examples["import_weight_kg"]
    ),
)

mirror_examples["calculated_mirror_factor"] = (
    mirror_examples["calculated_import_unit_value"]
    / mirror_examples["calculated_export_unit_value"]
)

mirror_examples[
    [
        "year",
        "exporter_iso",
        "importer_iso",
        "target_cmd",
        "export_value_fob",
        "import_value_cif",
        "export_weight_kg",
        "import_weight_kg",
        "weight_concordance",
        "calculated_export_unit_value",
        "calculated_import_unit_value",
        "calculated_mirror_factor",
        "mirror_unit_value_factor",
    ]
]

,year,exporter_iso,importer_iso,target_cmd,export_value_fob,import_value_cif,export_weight_kg,import_weight_kg,weight_concordance,calculated_export_unit_value,calculated_import_unit_value,calculated_mirror_factor,mirror_unit_value_factor
0,2023,CHN,GBR,491191,1.322924e+07,1.584340e+07,2.848463e+06,2.962048e+06,0.961653,4.644345,5.348801,1.151680,1.151680
1,2024,POL,BIH,480258,4.821900e+04,5.309173e+04,4.150000e+04,4.210356e+04,0.985665,1.161904,1.260980,1.085270,1.085270
2,2018,RUS,AZE,480257,1.332932e+06,1.338436e+06,1.461943e+06,1.446110e+06,0.989170,0.911754,0.925543,1.015123,1.015123
3,2019,TUR,HUN,940161,7.676310e+05,9.132710e+05,1.060120e+05,1.091920e+05,0.970877,7.240982,8.363900,1.155078,1.155078
4,2019,CHE,CRI,491000,4.678920e+02,7.371400e+02,3.900000e+01,4.090100e+01,0.953522,11.997231,18.022542,1.502225,1.502225
5,2019,BRA,IND,480100,8.545000e+04,1.052763e+05,2.034530e+05,2.030000e+05,0.997773,0.419999,0.518603,1.234772,1.234772
6,2022,SWE,SGP,480100,2.744840e+06,3.116558e+06,3.680000e+06,3.815650e+06,0.964449,0.745880,0.816783,1.095059,1.095059
7,2018,LBN,AUT,481910,4.200000e+01,1.709029e+03,2.504700e+01,2.500000e+01,0.998124,1.676848,68.361160,40.767666,40.767666
8,2019,LUX,DEU,481013,1.870544e+05,1.881825e+05,9.135319e+04,8.883400e+04,0.972424,2.047596,2.118362,1.034560,1.034560
9,2021,DEU,MMR,481200,2.055515e+04,2.351414e+04,1.836000e+03,1.971625e+03,0.931212,11.195614,11.926274,1.065263,1.065263


## Inspect Direct-Factor Cases

In [6]:
near_threshold = duckdb.sql(
    f"""
    SELECT
        year,
        exporter_iso,
        importer_iso,
        target_cmd,

        export_value_fob,
        import_value_cif,

        export_weight_kg,
        import_weight_kg,
        weight_concordance,

        export_unit_value_usd_per_kg,
        import_unit_value_usd_per_kg,
        mirror_unit_value_factor

    FROM read_parquet('{training_path}')

    WHERE unit_value_pair_eligible
      AND weight_concordance BETWEEN 0.895 AND 0.905

    ORDER BY weight_concordance
    LIMIT 30
    """
).df()

near_threshold

,year,exporter_iso,importer_iso,target_cmd,export_value_fob,import_value_cif,export_weight_kg,import_weight_kg,weight_concordance,export_unit_value_usd_per_kg,import_unit_value_usd_per_kg,mirror_unit_value_factor
0,2020,NLD,HRV,470421,4.569000e+00,6.000000e+00,2.000000e+00,1.790000e+00,0.895000,2.284500,3.351955,1.467260
1,2018,CAN,IRL,490890,6.484300e+01,2.102100e+02,8.950000e-01,1.000000e+00,0.895000,72.450279,210.210000,2.901438
2,2022,GRC,CYP,480591,2.369360e+02,3.580370e+02,2.000000e+02,1.790000e+02,0.895000,1.184680,2.000207,1.688394
3,2021,SAU,ARE,482361,9.645330e+02,6.695710e+02,3.580000e+02,4.000000e+02,0.895000,2.694226,1.673928,0.621302
4,2020,ITA,IRN,391239,8.374581e+03,4.727000e+03,2.000000e+02,1.790000e+02,0.895000,41.872905,26.407821,0.630666
5,2019,HUN,GRC,490890,1.140000e+02,1.119000e+00,8.950000e-01,1.000000e+00,0.895000,127.374302,1.119000,0.008785
6,2022,LTU,ARE,491000,4.510000e+02,6.014190e+02,6.000000e+01,5.370000e+01,0.895000,7.516667,11.199609,1.489970
7,2017,CZE,HUN,480257,2.234937e+06,2.394711e+06,1.666030e+06,1.861484e+06,0.895001,1.341475,1.286453,0.958984
8,2022,SWE,USA,392490,2.968188e+06,2.614010e+06,1.850925e+05,2.068070e+05,0.895001,16.036243,12.639853,0.788205
9,2017,FRA,JPN,481830,2.995253e+04,3.621705e+04,4.100000e+03,4.581000e+03,0.895001,7.305496,7.905928,1.082189


## Modified Direct Factor Example 

In [8]:
import duckdb

prepared_path = (
    "/storage/vacoda/data/interim/cif_fob/target_h5/prepared/"
    "comtrade_target_h5_cif_fob_prepared.parquet"
)

valid_direct_examples = duckdb.sql(
    f"""
    WITH import_rows AS (
        SELECT
            year,
            partner_iso AS exporter_iso,
            reporter_iso AS importer_iso,
            target_cmd,

            primary_value_hs_synced,

            CASE
                WHEN cif_value_usd > 0
                 AND fob_value_usd > 0
                THEN cif_value_usd * allocation_weight
            END AS paired_cif_value,

            CASE
                WHEN cif_value_usd > 0
                 AND fob_value_usd > 0
                THEN fob_value_usd * allocation_weight
            END AS paired_fob_value,

            CASE
                WHEN cif_value_usd > 0
                 AND fob_value_usd > 0
                THEN 1
                ELSE 0
            END AS has_paired_values

        FROM read_parquet('{prepared_path}')

        WHERE flow_code = 'M'
    ),

    aggregated AS (
        SELECT
            year,
            exporter_iso,
            importer_iso,
            target_cmd,

            COUNT(*) AS import_source_rows,

            SUM(has_paired_values)
                AS paired_source_rows,

            SUM(primary_value_hs_synced)
                AS total_import_value_cif,

            SUM(paired_cif_value)
                AS paired_cif_value,

            SUM(paired_fob_value)
                AS paired_fob_value

        FROM import_rows

        GROUP BY
            year,
            exporter_iso,
            importer_iso,
            target_cmd
    ),

    calculated AS (
        SELECT
            *,

            paired_cif_value / paired_fob_value
                AS direct_cif_fob_factor,

            paired_cif_value / total_import_value_cif
                AS direct_factor_value_coverage,

            paired_source_rows::DOUBLE
                / import_source_rows
                AS direct_factor_row_coverage

        FROM aggregated

        WHERE paired_cif_value > 0
          AND paired_fob_value > 0
          AND total_import_value_cif > 0
    )

    SELECT *
    FROM calculated

    WHERE direct_factor_value_coverage >= 0.90
      AND direct_cif_fob_factor BETWEEN 1.0 AND 1.5

    ORDER BY RANDOM()
    LIMIT 20
    """
).df()

valid_direct_examples

,year,exporter_iso,importer_iso,target_cmd,import_source_rows,paired_source_rows,total_import_value_cif,paired_cif_value,paired_fob_value,direct_cif_fob_factor,direct_factor_value_coverage,direct_factor_row_coverage
0,2017,ITA,PER,481031,4,4.0,3.784629e+05,3.784629e+05,3.635356e+05,1.041062,1.0,1.0
1,2021,CYP,SRB,940390,8,8.0,9.600000e+01,9.600000e+01,9.600000e+01,1.000000,1.0,1.0
2,2024,TUR,ROU,480592,4,4.0,1.070160e+05,1.070160e+05,1.027025e+05,1.042000,1.0,1.0
3,2024,CAN,PER,480519,4,4.0,4.865206e+05,4.865206e+05,3.957013e+05,1.229515,1.0,1.0
4,2017,SLV,HND,481149,2,2.0,1.233130e+04,1.233130e+04,1.147006e+04,1.075086,1.0,1.0
5,2021,ITA,PHL,551613,1,1.0,7.313000e+03,7.313000e+03,7.122000e+03,1.026818,1.0,1.0
6,2018,MKD,SRB,470790,8,8.0,8.098256e+06,8.098256e+06,8.049672e+06,1.006036,1.0,1.0
7,2021,W00,ECU,540500,8,8.0,2.799592e+04,2.799592e+04,2.521440e+04,1.110315,1.0,1.0
8,2019,AUT,COL,480100,12,12.0,2.334339e+05,2.334339e+05,2.089349e+05,1.117257,1.0,1.0
9,2022,DEU,CHL,481032,1,1.0,3.820530e+04,3.820530e+04,3.404106e+04,1.122330,1.0,1.0


In [9]:
example = valid_direct_examples.iloc[0]

source_rows = duckdb.sql(
    f"""
    SELECT *
    FROM read_parquet('{prepared_path}')
    WHERE flow_code = 'M'
      AND year = {int(example["year"])}
      AND reporter_iso = '{example["importer_iso"]}'
      AND partner_iso = '{example["exporter_iso"]}'
      AND target_cmd = '{example["target_cmd"]}'
    """
).df()

source_rows

,hs_sync_input_row_id,type_code,frequency_code,ref_period_id,year,month,period,reporter_code,reporter_iso,reporter_name,...,hs_sync_method,allocation_strategy,allocation_is_assumption,primary_value_original,primary_value_hs_synced,canonical_weight_kg,canonical_weight_source,canonical_weight_estimated,canonical_weight_is_proxy,canonical_weight_hs_synced
0,21455428,C,A,20170101,2017,52,2017,604,PER,Peru,...,already_target,deterministic,False,94615.73,94615.73,50630.0,reported_net_weight_kg,False,False,50630.0
1,21455431,C,A,20170101,2017,52,2017,604,PER,Peru,...,already_target,deterministic,False,94615.73,94615.73,50630.0,reported_net_weight_kg,False,False,50630.0
2,21455432,C,A,20170101,2017,52,2017,604,PER,Peru,...,already_target,deterministic,False,94615.73,94615.73,50630.0,reported_net_weight_kg,False,False,50630.0
3,21455567,C,A,20170101,2017,52,2017,604,PER,Peru,...,already_target,deterministic,False,94615.73,94615.73,50630.0,reported_net_weight_kg,False,False,50630.0


In [10]:
varying_columns = [
    column
    for column in source_rows.columns
    if source_rows[column].nunique(dropna=False) > 1
]

varying_columns


['hs_sync_input_row_id',
 'partner2_code',
 'partner2_iso',
 'partner2_name',
 'mot_code',
 'mot_name',
 'is_reported',
 'is_aggregate']

In [11]:
columns_to_show = [
    "year",
    "reporter_iso",
    "partner_iso",
    "flow_code",
    "target_cmd",
    "cif_value_usd",
    "fob_value_usd",
    "primary_value_hs_synced",
    "allocation_weight",
] + [
    column
    for column in varying_columns
    if column not in {
        "year",
        "reporter_iso",
        "partner_iso",
        "flow_code",
        "target_cmd",
        "cif_value_usd",
        "fob_value_usd",
        "primary_value_hs_synced",
        "allocation_weight",
    }
]

source_rows[columns_to_show]

,year,reporter_iso,partner_iso,flow_code,target_cmd,cif_value_usd,fob_value_usd,primary_value_hs_synced,allocation_weight,hs_sync_input_row_id,partner2_code,partner2_iso,partner2_name,mot_code,mot_name,is_reported,is_aggregate
0,2017,PER,ITA,M,481031,94615.73,90883.9,94615.73,1.0,21455428,380,ITA,Italy,2100,Sea,True,False
1,2017,PER,ITA,M,481031,94615.73,90883.9,94615.73,1.0,21455431,0,W00,World,2100,Sea,False,True
2,2017,PER,ITA,M,481031,94615.73,90883.9,94615.73,1.0,21455432,0,W00,World,0,TOTAL MOT,False,True
3,2017,PER,ITA,M,481031,94615.73,90883.9,94615.73,1.0,21455567,380,ITA,Italy,0,TOTAL MOT,False,True


In [13]:
check = duckdb.sql(
    f"""
    SELECT
        year,
        reporter_iso,
        partner_iso,
        flow_code,
        target_cmd,
        partner2_iso,
        mot_code,
        mot_name,
        is_reported,
        is_aggregate,
        cif_value_usd,
        fob_value_usd
    FROM read_parquet('{prepared_path}')
    WHERE year = 2017
      AND reporter_iso = 'PER'
      AND partner_iso = 'ITA'
      AND target_cmd = 481031
    ORDER BY
        mot_code,
        partner2_iso
    """
).df()

check

,year,reporter_iso,partner_iso,flow_code,target_cmd,partner2_iso,mot_code,mot_name,is_reported,is_aggregate,cif_value_usd,fob_value_usd
0,2017,PER,ITA,M,481031,ITA,0,TOTAL MOT,False,True,94615.73,90883.9
1,2017,PER,ITA,M,481031,W00,0,TOTAL MOT,False,True,94615.73,90883.9
2,2017,PER,ITA,M,481031,ITA,2100,Sea,True,False,94615.73,90883.9
3,2017,PER,ITA,M,481031,W00,2100,Sea,False,True,94615.73,90883.9


In [2]:
import duckdb

prepared_path = (
    "/storage/vacoda/data/interim/cif_fob/target_h5/prepared/"
    "comtrade_target_h5_cif_fob_prepared.parquet"
)
rows_per_group = duckdb.sql(
    f"""
    SELECT
        n_rows_per_group,
        COUNT(*) AS n_groups
    FROM (
        SELECT
            year,
            reporter_iso,
            partner_iso,
            flow_code,
            target_cmd,
            COUNT(*) AS n_rows_per_group
        FROM read_parquet('{prepared_path}')
        GROUP BY
            year,
            reporter_iso,
            partner_iso,
            flow_code,
            target_cmd
    )
    GROUP BY n_rows_per_group
    ORDER BY n_rows_per_group
    """
).df()

rows_per_group

,n_rows_per_group,n_groups
0,1,3928944
1,2,1062914
2,3,297218
3,4,1152085
4,5,28871
...,...,...
857,1240,1
858,1258,1
859,1324,1
860,1338,1


In [3]:
reported_vs_aggregate = duckdb.sql(
    f"""
    SELECT
        is_reported,
        is_aggregate,
        COUNT(*) AS n_rows,
        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            2
        ) AS share_percent
    FROM read_parquet('{prepared_path}')
    GROUP BY
        is_reported,
        is_aggregate
    ORDER BY n_rows DESC
    """
).df()

reported_vs_aggregate

,is_reported,is_aggregate,n_rows,share_percent
0,False,True,28068145,70.36
1,True,False,11822447,29.64


In [4]:
overall_reduction = duckdb.sql(
    f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE is_reported = TRUE
              AND is_aggregate = FALSE
        ) AS reported_detail_rows,

        ROUND(
            100.0 *
            COUNT(*) FILTER (
                WHERE is_reported = TRUE
                  AND is_aggregate = FALSE
            )
            / COUNT(*),
            2
        ) AS reported_detail_share_percent,

        ROUND(
            COUNT(*)::DOUBLE
            /
            NULLIF(
                COUNT(*) FILTER (
                    WHERE is_reported = TRUE
                      AND is_aggregate = FALSE
                ),
                0
            ),
            2
        ) AS total_to_reported_row_ratio

    FROM read_parquet('{prepared_path}')
    """
).df()

overall_reduction

,total_rows,reported_detail_rows,reported_detail_share_percent,total_to_reported_row_ratio
0,39890592,11822447,29.64,3.37


In [5]:
dimension_patterns = duckdb.sql(
    f"""
    SELECT
        is_reported,
        is_aggregate,
        partner2_iso,
        mot_code,
        mot_name,
        COUNT(*) AS n_rows

    FROM read_parquet('{prepared_path}')

    GROUP BY
        is_reported,
        is_aggregate,
        partner2_iso,
        mot_code,
        mot_name

    ORDER BY n_rows DESC

    LIMIT 30
    """
).df()

dimension_patterns

,is_reported,is_aggregate,partner2_iso,mot_code,mot_name,n_rows
0,False,True,W00,0,TOTAL MOT,6896526
1,True,False,W00,0,TOTAL MOT,3802568
2,False,True,W00,3200,Road,2188126
3,False,True,W00,1000,Air,1676399
4,False,True,W00,2100,Sea,1584246
5,False,True,_X,0,TOTAL MOT,1207224
6,True,False,W00,3200,Road,633782
7,True,False,W00,2100,Sea,624851
8,True,False,W00,1000,Air,624494
9,False,True,W00,9900,Other,542533


In [6]:
duplicate_value_patterns = duckdb.sql(
    f"""
    WITH grouped AS (
        SELECT
            year,
            reporter_iso,
            partner_iso,
            flow_code,
            target_cmd,

            COUNT(*) AS n_rows,

            COUNT(DISTINCT primary_value_hs_synced)
                AS n_distinct_primary_values,

            COUNT(DISTINCT canonical_weight_hs_synced)
                AS n_distinct_weights,

            MIN(primary_value_hs_synced)
                AS min_value,

            MAX(primary_value_hs_synced)
                AS max_value

        FROM read_parquet('{prepared_path}')

        GROUP BY
            year,
            reporter_iso,
            partner_iso,
            flow_code,
            target_cmd
    )

    SELECT
        n_rows,
        n_distinct_primary_values,
        n_distinct_weights,
        COUNT(*) AS n_groups

    FROM grouped

    GROUP BY
        n_rows,
        n_distinct_primary_values,
        n_distinct_weights

    ORDER BY n_groups DESC

    LIMIT 30
    """
).df()

duplicate_value_patterns

,n_rows,n_distinct_primary_values,n_distinct_weights,n_groups
0,1,1,1,3891511
1,4,1,1,1045766
2,2,1,1,1016963
3,8,1,1,594024
4,6,3,3,332084
5,3,3,3,287674
6,12,3,3,220024
7,4,4,4,89687
8,8,4,4,81035
9,16,4,4,54248


In [7]:
value_inflation = duckdb.sql(
    f"""
    SELECT
        SUM(primary_value_hs_synced)
            AS value_all_rows,

        SUM(primary_value_hs_synced) FILTER (
            WHERE is_reported = TRUE
              AND is_aggregate = FALSE
        ) AS value_reported_detail_only,

        SUM(primary_value_hs_synced)
        /
        NULLIF(
            SUM(primary_value_hs_synced) FILTER (
                WHERE is_reported = TRUE
                  AND is_aggregate = FALSE
            ),
            0
        ) AS apparent_value_inflation_ratio

    FROM read_parquet('{prepared_path}')
    """
).df()

value_inflation

,value_all_rows,value_reported_detail_only,apparent_value_inflation_ratio
0,4.820888e+13,9.331897e+12,5.166032


In [8]:
canonical_total_availability = duckdb.sql(
    f"""
    WITH groups AS (
        SELECT
            year,
            reporter_iso,
            partner_iso,
            flow_code,
            target_cmd,

            COUNT(*) AS n_rows,

            COUNT(*) FILTER (
                WHERE partner2_iso = 'W00'
                  AND mot_code = 0
            ) AS n_world_total_mot_rows,

            COUNT(*) FILTER (
                WHERE partner2_iso = 'W00'
                  AND mot_code = 0
                  AND is_reported = TRUE
                  AND is_aggregate = FALSE
            ) AS n_reported_world_total_mot_rows,

            COUNT(*) FILTER (
                WHERE is_reported = TRUE
                  AND is_aggregate = FALSE
            ) AS n_reported_rows

        FROM read_parquet('{prepared_path}')

        GROUP BY
            year,
            reporter_iso,
            partner_iso,
            flow_code,
            target_cmd
    )

    SELECT
        CASE
            WHEN n_reported_world_total_mot_rows > 0
                THEN 'reported_world_total_mot'
            WHEN n_world_total_mot_rows > 0
                THEN 'aggregate_world_total_mot'
            WHEN n_reported_rows > 0
                THEN 'must_build_from_reported_detail'
            ELSE 'no_reported_source'
        END AS canonical_source_candidate,

        COUNT(*) AS n_groups

    FROM groups

    GROUP BY 1
    ORDER BY n_groups DESC
    """
).df()

canonical_total_availability

,canonical_source_candidate,n_groups
0,aggregate_world_total_mot,4633595
1,reported_world_total_mot,3748387


## Non Agregated, only reported 

In [9]:
import duckdb

prepared_path = (
    "/storage/vacoda/data/interim/cif_fob/target_h5/prepared/"
    "comtrade_target_h5_cif_fob_prepared.parquet"
)

reported_group_structure = duckdb.sql(
    f"""
    WITH groups AS (
        SELECT
            year,
            reporter_iso,
            partner_iso,
            flow_code,
            target_cmd,

            COUNT(*) AS n_reported_rows,

            COUNT(DISTINCT partner2_iso)
                AS n_partner2_values,

            COUNT(DISTINCT mot_code)
                AS n_mot_values,

            COUNT(DISTINCT primary_value_hs_synced)
                AS n_distinct_values,

            COUNT(DISTINCT canonical_weight_hs_synced)
                AS n_distinct_weights

        FROM read_parquet('{prepared_path}')

        GROUP BY
            year,
            reporter_iso,
            partner_iso,
            flow_code,
            target_cmd
    )

    SELECT
        n_reported_rows,
        COUNT(*) AS n_groups

    FROM groups

    GROUP BY n_reported_rows
    ORDER BY n_reported_rows
    """
).df()

reported_group_structure

,n_reported_rows,n_groups
0,1,6268973
1,2,907253
2,3,300736
3,4,122168
4,5,61952
...,...,...
166,206,1
167,207,1
168,209,1
169,210,1


In [10]:
reported_group_summary = duckdb.sql(
    f"""
    WITH groups AS (
        SELECT
            year,
            reporter_iso,
            partner_iso,
            flow_code,
            target_cmd,

            COUNT(*) AS n_reported_rows,

            COUNT(DISTINCT partner2_iso)
                AS n_partner2_values,

            COUNT(DISTINCT mot_code)
                AS n_mot_values

        FROM read_parquet('{prepared_path}')

        GROUP BY
            year,
            reporter_iso,
            partner_iso,
            flow_code,
            target_cmd
    )

    SELECT
        COUNT(*) AS n_canonical_groups,

        COUNT(*) FILTER (
            WHERE n_reported_rows = 1
        ) AS n_single_row_groups,

        COUNT(*) FILTER (
            WHERE n_reported_rows > 1
        ) AS n_multi_row_groups,

        COUNT(*) FILTER (
            WHERE n_partner2_values > 1
        ) AS n_multi_partner2_groups,

        COUNT(*) FILTER (
            WHERE n_mot_values > 1
        ) AS n_multi_mot_groups,

        MAX(n_reported_rows)
            AS max_reported_rows_per_group

    FROM groups
    """
).df()

reported_group_summary

,n_canonical_groups,n_single_row_groups,n_multi_row_groups,n_multi_partner2_groups,n_multi_mot_groups,max_reported_rows_per_group
0,7831576,6268973,1562603,573817,1256969,214


In [11]:
multirow_structure = duckdb.sql(
    f"""
    WITH groups AS (
        SELECT
            year,
            reporter_iso,
            partner_iso,
            flow_code,
            target_cmd,

            COUNT(*) AS n_reported_rows,

            COUNT(*) FILTER (
                WHERE mot_code = 0
            ) AS n_total_mot_rows,

            COUNT(*) FILTER (
                WHERE mot_code <> 0
            ) AS n_detail_mot_rows,

            COUNT(*) FILTER (
                WHERE partner2_iso = 'W00'
            ) AS n_world_partner2_rows,

            COUNT(*) FILTER (
                WHERE partner2_iso <> 'W00'
            ) AS n_detail_partner2_rows

        FROM read_parquet('{prepared_path}')

        GROUP BY
            year,
            reporter_iso,
            partner_iso,
            flow_code,
            target_cmd
    )

    SELECT
        CASE
            WHEN n_reported_rows = 1
                THEN 'single_reported_row'

            WHEN n_total_mot_rows > 0
             AND n_detail_mot_rows > 0
             AND n_world_partner2_rows > 0
             AND n_detail_partner2_rows > 0
                THEN 'overlap_mot_and_partner2'

            WHEN n_total_mot_rows > 0
             AND n_detail_mot_rows > 0
                THEN 'overlap_mot'

            WHEN n_world_partner2_rows > 0
             AND n_detail_partner2_rows > 0
                THEN 'overlap_partner2'

            WHEN n_total_mot_rows = 0
             AND n_world_partner2_rows = 0
                THEN 'detail_only'

            ELSE 'other_structure'
        END AS structure_class,

        COUNT(*) AS n_groups

    FROM groups

    GROUP BY 1
    ORDER BY n_groups DESC
    """
).df()

multirow_structure

,structure_class,n_groups
0,single_reported_row,6268973
1,detail_only,933712
2,other_structure,628891


In [12]:
mot_reconciliation = duckdb.sql(
    f"""
    WITH grouped AS (
        SELECT
            year,
            reporter_iso,
            partner_iso,
            flow_code,
            target_cmd,
            partner2_iso,

            SUM(
                CASE
                    WHEN mot_code = 0
                    THEN primary_value_hs_synced
                END
            ) AS total_mot_value,

            SUM(
                CASE
                    WHEN mot_code <> 0
                    THEN primary_value_hs_synced
                ELSE 0
                END
            ) AS detail_mot_value

        FROM read_parquet('{prepared_path}')

        GROUP BY
            year,
            reporter_iso,
            partner_iso,
            flow_code,
            target_cmd,
            partner2_iso
    )

    SELECT
        COUNT(*) FILTER (
            WHERE total_mot_value IS NOT NULL
              AND detail_mot_value > 0
        ) AS n_comparable_groups,

        MEDIAN(
            total_mot_value / detail_mot_value
        ) FILTER (
            WHERE total_mot_value > 0
              AND detail_mot_value > 0
        ) AS median_total_to_detail_ratio,

        COUNT(*) FILTER (
            WHERE total_mot_value > 0
              AND detail_mot_value > 0
              AND ABS(
                  total_mot_value / detail_mot_value - 1
              ) <= 0.01
        ) AS n_within_1_percent

    FROM grouped
    """
).df()

mot_reconciliation

,n_comparable_groups,median_total_to_detail_ratio,n_within_1_percent
0,0,NaN,0


In [13]:
partner2_reconciliation = duckdb.sql(
    f"""
    WITH grouped AS (
        SELECT
            year,
            reporter_iso,
            partner_iso,
            flow_code,
            target_cmd,
            mot_code,

            SUM(
                CASE
                    WHEN partner2_iso = 'W00'
                    THEN primary_value_hs_synced
                END
            ) AS world_partner2_value,

            SUM(
                CASE
                    WHEN partner2_iso <> 'W00'
                    THEN primary_value_hs_synced
                ELSE 0
                END
            ) AS detail_partner2_value

        FROM read_parquet('{prepared_path}')

        GROUP BY
            year,
            reporter_iso,
            partner_iso,
            flow_code,
            target_cmd,
            mot_code
    )

    SELECT
        COUNT(*) FILTER (
            WHERE world_partner2_value IS NOT NULL
              AND detail_partner2_value > 0
        ) AS n_comparable_groups,

        MEDIAN(
            world_partner2_value / detail_partner2_value
        ) FILTER (
            WHERE world_partner2_value > 0
              AND detail_partner2_value > 0
        ) AS median_world_to_detail_ratio,

        COUNT(*) FILTER (
            WHERE world_partner2_value > 0
              AND detail_partner2_value > 0
              AND ABS(
                  world_partner2_value / detail_partner2_value - 1
              ) <= 0.01
        ) AS n_within_1_percent

    FROM grouped
    """
).df()

partner2_reconciliation

,n_comparable_groups,median_world_to_detail_ratio,n_within_1_percent
0,0,NaN,0


In [14]:
reported_structure_detail = duckdb.sql(
    f"""
    WITH groups AS (
        SELECT
            year,
            reporter_iso,
            partner_iso,
            flow_code,
            target_cmd,

            COUNT(*) AS n_reported_rows,

            COUNT(*) FILTER (
                WHERE mot_code = 0
            ) AS n_mot_total_rows,

            COUNT(*) FILTER (
                WHERE mot_code <> 0
            ) AS n_mot_detail_rows,

            COUNT(*) FILTER (
                WHERE partner2_iso = 'W00'
            ) AS n_partner2_total_rows,

            COUNT(*) FILTER (
                WHERE partner2_iso <> 'W00'
            ) AS n_partner2_detail_rows,

            COUNT(DISTINCT mot_code)
                AS n_distinct_mot,

            COUNT(DISTINCT partner2_iso)
                AS n_distinct_partner2

        FROM read_parquet('{prepared_path}')

        GROUP BY
            year,
            reporter_iso,
            partner_iso,
            flow_code,
            target_cmd
    )

    SELECT
        CASE
            WHEN n_reported_rows = 1
                THEN 'single_row'

            WHEN n_mot_total_rows = 0
             AND n_partner2_total_rows = 0
                THEN 'partner2_detail_x_mot_detail'

            WHEN n_mot_total_rows = 0
             AND n_partner2_detail_rows = 0
                THEN 'partner2_total_x_mot_detail'

            WHEN n_mot_detail_rows = 0
             AND n_partner2_total_rows = 0
                THEN 'partner2_detail_x_mot_total'

            WHEN n_mot_detail_rows = 0
             AND n_partner2_detail_rows = 0
                THEN 'partner2_total_x_mot_total'

            WHEN n_mot_total_rows > 0
             AND n_mot_detail_rows > 0
                THEN 'mixed_mot_levels'

            WHEN n_partner2_total_rows > 0
             AND n_partner2_detail_rows > 0
                THEN 'mixed_partner2_levels'

            ELSE 'unclassified'
        END AS structure_class,

        COUNT(*) AS n_groups,

        SUM(n_reported_rows) AS n_rows,

        MIN(n_reported_rows) AS min_rows_per_group,

        MEDIAN(n_reported_rows) AS median_rows_per_group,

        MAX(n_reported_rows) AS max_rows_per_group

    FROM groups

    GROUP BY 1
    ORDER BY n_groups DESC
    """
).df()

reported_structure_detail

,structure_class,n_groups,n_rows,min_rows_per_group,median_rows_per_group,max_rows_per_group
0,single_row,6268973,6268973.0,1,1.0,1
1,partner2_detail_x_mot_detail,933712,3965749.0,2,3.0,214
2,partner2_total_x_mot_detail,492865,1136108.0,2,2.0,15
3,partner2_detail_x_mot_total,92597,354007.0,2,3.0,46
4,partner2_total_x_mot_total,43429,97610.0,2,2.0,8


In [15]:
multirow_varying_columns = duckdb.sql(
    f"""
    WITH multi_keys AS (
        SELECT
            year,
            reporter_iso,
            partner_iso,
            flow_code,
            target_cmd
        FROM read_parquet('{prepared_path}')
        GROUP BY
            year,
            reporter_iso,
            partner_iso,
            flow_code,
            target_cmd
        HAVING COUNT(*) > 1
    )

    SELECT p.*
    FROM read_parquet('{prepared_path}') AS p
    INNER JOIN multi_keys AS k
        USING (
            year,
            reporter_iso,
            partner_iso,
            flow_code,
            target_cmd
        )
    LIMIT 10000
    """
).df()

varying_columns = [
    column
    for column in multirow_varying_columns.columns
    if multirow_varying_columns[column].nunique(
        dropna=False
    ) > 1
]

varying_columns

['hs_sync_input_row_id',
 'ref_period_id',
 'year',
 'period',
 'reporter_code',
 'reporter_iso',
 'reporter_name',
 'flow_code',
 'flow_name',
 'partner_code',
 'partner_iso',
 'partner_name',
 'partner2_code',
 'partner2_iso',
 'partner2_name',
 'source_hs',
 'source_cmd',
 'commodity_name',
 'customs_code',
 'customs_name',
 'mos_code',
 'mot_code',
 'mot_name',
 'quantity_unit_code',
 'quantity_unit',
 'quantity',
 'is_quantity_estimated',
 'alt_quantity_unit_code',
 'alt_quantity_unit',
 'alt_quantity',
 'net_weight_kg',
 'is_net_weight_estimated',
 'gross_weight_kg',
 'cif_value_usd',
 'fob_value_usd',
 'primary_value',
 'legacy_estimation_flag',
 'source_hs_original',
 'source_cmd_original',
 'target_cmd',
 'allocation_weight',
 'conversion_need',
 'hs_sync_method',
 'allocation_strategy',
 'allocation_is_assumption',
 'primary_value_original',
 'primary_value_hs_synced',
 'canonical_weight_kg',
 'canonical_weight_source',
 'canonical_weight_estimated',
 'canonical_weight_is_pro

### Inspect one of each multi-row cases

In [16]:
structure_groups = duckdb.sql(
    f"""
    WITH groups AS (
        SELECT
            year,
            reporter_iso,
            partner_iso,
            flow_code,
            target_cmd,

            COUNT(*) AS n_reported_rows,

            COUNT(*) FILTER (
                WHERE mot_code = 0
            ) AS n_mot_total_rows,

            COUNT(*) FILTER (
                WHERE mot_code <> 0
            ) AS n_mot_detail_rows,

            COUNT(*) FILTER (
                WHERE partner2_iso = 'W00'
            ) AS n_partner2_total_rows,

            COUNT(*) FILTER (
                WHERE partner2_iso <> 'W00'
            ) AS n_partner2_detail_rows

        FROM read_parquet('{prepared_path}')

        GROUP BY
            year,
            reporter_iso,
            partner_iso,
            flow_code,
            target_cmd
    )

    SELECT
        *,

        CASE
            WHEN n_reported_rows = 1
                THEN 'single_row'

            WHEN n_mot_total_rows = 0
             AND n_partner2_total_rows = 0
                THEN 'partner2_detail_x_mot_detail'

            WHEN n_mot_total_rows = 0
             AND n_partner2_detail_rows = 0
                THEN 'partner2_total_x_mot_detail'

            WHEN n_mot_detail_rows = 0
             AND n_partner2_total_rows = 0
                THEN 'partner2_detail_x_mot_total'

            WHEN n_mot_detail_rows = 0
             AND n_partner2_detail_rows = 0
                THEN 'partner2_total_x_mot_total'

            ELSE 'unclassified'
        END AS structure_class

    FROM groups
    """
).df()

structure_groups.head()

,year,reporter_iso,partner_iso,flow_code,target_cmd,n_reported_rows,n_mot_total_rows,n_mot_detail_rows,n_partner2_total_rows,n_partner2_detail_rows,structure_class
0,2017,PAK,KWT,X,481820,1,1,0,1,0,single_row
1,2017,PAK,AUS,X,481820,1,1,0,1,0,single_row
2,2017,NIC,TUR,M,481820,2,0,2,2,0,partner2_total_x_mot_detail
3,2017,NGA,NLD,M,481820,1,1,0,1,0,single_row
4,2017,NOR,EST,M,481820,2,2,0,0,2,partner2_detail_x_mot_total


In [17]:
example_keys = duckdb.sql(
    f"""
    WITH groups AS (
        SELECT
            year,
            reporter_iso,
            partner_iso,
            flow_code,
            target_cmd,

            COUNT(*) AS n_reported_rows,

            COUNT(*) FILTER (
                WHERE mot_code = 0
            ) AS n_mot_total_rows,

            COUNT(*) FILTER (
                WHERE mot_code <> 0
            ) AS n_mot_detail_rows,

            COUNT(*) FILTER (
                WHERE partner2_iso = 'W00'
            ) AS n_partner2_total_rows,

            COUNT(*) FILTER (
                WHERE partner2_iso <> 'W00'
            ) AS n_partner2_detail_rows

        FROM read_parquet('{prepared_path}')

        GROUP BY
            year,
            reporter_iso,
            partner_iso,
            flow_code,
            target_cmd
    ),

    classified AS (
        SELECT
            *,

            CASE
                WHEN n_reported_rows = 1
                    THEN 'single_row'

                WHEN n_mot_total_rows = 0
                 AND n_partner2_total_rows = 0
                    THEN 'partner2_detail_x_mot_detail'

                WHEN n_mot_total_rows = 0
                 AND n_partner2_detail_rows = 0
                    THEN 'partner2_total_x_mot_detail'

                WHEN n_mot_detail_rows = 0
                 AND n_partner2_total_rows = 0
                    THEN 'partner2_detail_x_mot_total'

                WHEN n_mot_detail_rows = 0
                 AND n_partner2_detail_rows = 0
                    THEN 'partner2_total_x_mot_total'

                ELSE 'unclassified'
            END AS structure_class

        FROM groups
    ),

    ranked AS (
        SELECT
            *,

            ROW_NUMBER() OVER (
                PARTITION BY structure_class
                ORDER BY
                    ABS(n_reported_rows - 3),
                    year DESC
            ) AS example_rank

        FROM classified

        WHERE structure_class IN (
            'single_row',
            'partner2_detail_x_mot_detail',
            'partner2_total_x_mot_detail',
            'partner2_detail_x_mot_total',
            'partner2_total_x_mot_total'
        )
    )

    SELECT
        structure_class,
        year,
        reporter_iso,
        partner_iso,
        flow_code,
        target_cmd,
        n_reported_rows

    FROM ranked

    WHERE example_rank = 1

    ORDER BY structure_class
    """
).df()

example_keys

,structure_class,year,reporter_iso,partner_iso,flow_code,target_cmd,n_reported_rows
0,partner2_detail_x_mot_detail,2025,LVA,LTU,X,440910,3
1,partner2_detail_x_mot_total,2025,FIN,FRA,X,481013,3
2,partner2_total_x_mot_detail,2025,MYS,PHL,X,482190,3
3,partner2_total_x_mot_total,2025,ITA,HUN,X,441299,3
4,single_row,2025,AUS,PAK,M,482010,1


In [19]:
import pandas as pd
def inspect_group(example_row: pd.Series) -> pd.DataFrame:
    year = int(example_row["year"])
    reporter = example_row["reporter_iso"]
    partner = example_row["partner_iso"]
    flow = example_row["flow_code"]
    target_cmd = example_row["target_cmd"]

    return duckdb.sql(
        f"""
        SELECT
            hs_sync_input_row_id,

            year,
            reporter_iso,
            partner_iso,
            flow_code,
            target_cmd,

            partner2_iso,
            partner2_name,

            mot_code,
            mot_name,

            customs_code,
            customs_name,
            mos_code,

            source_hs,
            source_cmd,
            source_cmd_original,

            primary_value_original,
            primary_value_hs_synced,
            allocation_weight,

            cif_value_usd,
            fob_value_usd,

            canonical_weight_hs_synced,
            canonical_weight_source,

            is_reported,
            is_aggregate

        FROM read_parquet('{prepared_path}')

        WHERE year = {year}
          AND reporter_iso = '{reporter}'
          AND partner_iso = '{partner}'
          AND flow_code = '{flow}'
          AND target_cmd = '{target_cmd}'

        ORDER BY
            partner2_iso,
            mot_code,
            customs_code,
            mos_code,
            source_cmd
        """
    ).df()

In [20]:
examples = {}

for _, example_row in example_keys.iterrows():
    structure_class = example_row["structure_class"]

    examples[structure_class] = inspect_group(
        example_row
    )

    print("=" * 100)
    print(structure_class)
    print(example_row.to_dict())
    display(examples[structure_class])

partner2_detail_x_mot_detail
{'structure_class': 'partner2_detail_x_mot_detail', 'year': 2025, 'reporter_iso': 'LVA', 'partner_iso': 'LTU', 'flow_code': 'X', 'target_cmd': '440910', 'n_reported_rows': 3}


,hs_sync_input_row_id,year,reporter_iso,partner_iso,flow_code,target_cmd,partner2_iso,partner2_name,mot_code,mot_name,...,source_cmd_original,primary_value_original,primary_value_hs_synced,allocation_weight,cif_value_usd,fob_value_usd,canonical_weight_hs_synced,canonical_weight_source,is_reported,is_aggregate
0,5476794,2025,LVA,LTU,X,440910,_X,"Areas, nes",3200,Road,...,440910,593253.573,593253.573,1.0,NaN,593253.573,499244.0,reported_net_weight_kg,True,False
1,5476783,2025,LVA,LTU,X,440910,_X,"Areas, nes",9200,"Postal consignments, mail or courier shipment",...,440910,38.419,38.419,1.0,NaN,38.419,33.0,reported_net_weight_kg,True,False
2,5476786,2025,LVA,LTU,X,440910,_X,"Areas, nes",9900,Other,...,440910,55537.541,55537.541,1.0,NaN,55537.541,49879.0,reported_net_weight_kg,True,False


partner2_detail_x_mot_total
{'structure_class': 'partner2_detail_x_mot_total', 'year': 2025, 'reporter_iso': 'FIN', 'partner_iso': 'FRA', 'flow_code': 'X', 'target_cmd': '481013', 'n_reported_rows': 3}


,hs_sync_input_row_id,year,reporter_iso,partner_iso,flow_code,target_cmd,partner2_iso,partner2_name,mot_code,mot_name,...,source_cmd_original,primary_value_original,primary_value_hs_synced,allocation_weight,cif_value_usd,fob_value_usd,canonical_weight_hs_synced,canonical_weight_source,is_reported,is_aggregate
0,20828283,2025,FIN,FRA,X,481013,AUT,Austria,0,TOTAL MOT,...,481013,26973.365,26973.365,1.0,NaN,26973.365,16277.0,reported_net_weight_kg,True,False
1,20828284,2025,FIN,FRA,X,481013,DEU,Germany,0,TOTAL MOT,...,481013,3865.047,3865.047,1.0,NaN,3865.047,8221.0,reported_net_weight_kg,True,False
2,20828437,2025,FIN,FRA,X,481013,_X,"Areas, nes",0,TOTAL MOT,...,481013,4347369.476,4347369.476,1.0,NaN,4347369.476,2230704.0,reported_net_weight_kg,True,False


partner2_total_x_mot_detail
{'structure_class': 'partner2_total_x_mot_detail', 'year': 2025, 'reporter_iso': 'MYS', 'partner_iso': 'PHL', 'flow_code': 'X', 'target_cmd': '482190', 'n_reported_rows': 3}


,hs_sync_input_row_id,year,reporter_iso,partner_iso,flow_code,target_cmd,partner2_iso,partner2_name,mot_code,mot_name,...,source_cmd_original,primary_value_original,primary_value_hs_synced,allocation_weight,cif_value_usd,fob_value_usd,canonical_weight_hs_synced,canonical_weight_source,is_reported,is_aggregate
0,33547301,2025,MYS,PHL,X,482190,W00,World,1000,Air,...,482190,81358.890,81358.890,1.0,NaN,81358.890,731593.742,reported_net_weight_kg,True,False
1,33547369,2025,MYS,PHL,X,482190,W00,World,2000,Water,...,482190,37905.443,37905.443,1.0,NaN,37905.443,4483.565,reported_net_weight_kg,True,False
2,33547231,2025,MYS,PHL,X,482190,W00,World,3200,Road,...,482190,11711.259,11711.259,1.0,NaN,11711.259,343.800,reported_net_weight_kg,True,False


partner2_total_x_mot_total
{'structure_class': 'partner2_total_x_mot_total', 'year': 2025, 'reporter_iso': 'ITA', 'partner_iso': 'HUN', 'flow_code': 'X', 'target_cmd': '441299', 'n_reported_rows': 3}


,hs_sync_input_row_id,year,reporter_iso,partner_iso,flow_code,target_cmd,partner2_iso,partner2_name,mot_code,mot_name,...,source_cmd_original,primary_value_original,primary_value_hs_synced,allocation_weight,cif_value_usd,fob_value_usd,canonical_weight_hs_synced,canonical_weight_source,is_reported,is_aggregate
0,7776821,2025,ITA,HUN,X,441299,W00,World,0,TOTAL MOT,...,441291,1640.736,1640.736,1.0,NaN,1640.736,1555.0,reported_net_weight_kg,True,False
1,7806346,2025,ITA,HUN,X,441299,W00,World,0,TOTAL MOT,...,441292,41828.586,41828.586,1.0,NaN,41828.586,11729.0,reported_net_weight_kg,True,False
2,8035784,2025,ITA,HUN,X,441299,W00,World,0,TOTAL MOT,...,441299,83284.277,83284.277,1.0,NaN,83284.277,18767.0,reported_net_weight_kg,True,False


single_row
{'structure_class': 'single_row', 'year': 2025, 'reporter_iso': 'AUS', 'partner_iso': 'PAK', 'flow_code': 'M', 'target_cmd': '482010', 'n_reported_rows': 1}


,hs_sync_input_row_id,year,reporter_iso,partner_iso,flow_code,target_cmd,partner2_iso,partner2_name,mot_code,mot_name,...,source_cmd_original,primary_value_original,primary_value_hs_synced,allocation_weight,cif_value_usd,fob_value_usd,canonical_weight_hs_synced,canonical_weight_source,is_reported,is_aggregate
0,30790772,2025,AUS,PAK,M,482010,W00,World,2100,Sea,...,482010,2563.85,2563.85,1.0,2563.85,2401.24,936.288,estimated_quantity_kg,True,False


## Why mirror pairs are excluded

In [21]:
import duckdb

mirror_path = (
    "/storage/vacoda/data/interim/cif_fob/target_h5/"
    "target_h5_cif_fob_mirror_pairs.parquet"
)

exclusion_summary = duckdb.sql(
    f"""
    SELECT
        exclusion_reason,
        COUNT(*) AS n_groups,

        ROUND(
            100.0 * COUNT(*)
            / SUM(COUNT(*)) OVER (),
            2
        ) AS share_all_percent

    FROM read_parquet('{mirror_path}')

    GROUP BY exclusion_reason
    ORDER BY n_groups DESC
    """
).df()

exclusion_summary

,exclusion_reason,n_groups,share_all_percent
0,low_weight_concordance,1998780,36.42
1,missing_export_report,1831489,33.37
2,missing_import_report,1328738,24.21
3,eligible,304846,5.55
4,missing_or_nonpositive_import_weight,14409,0.26
5,missing_or_nonpositive_export_weight,9718,0.18
6,nonpositive_export_value,1,0.00


In [22]:
weight_concordance_summary = duckdb.sql(
    f"""
    SELECT
        COUNT(*) AS n_comparable_groups,

        QUANTILE_CONT(weight_concordance, 0.01) AS p01,
        QUANTILE_CONT(weight_concordance, 0.05) AS p05,
        QUANTILE_CONT(weight_concordance, 0.10) AS p10,
        QUANTILE_CONT(weight_concordance, 0.25) AS p25,
        MEDIAN(weight_concordance) AS median,
        QUANTILE_CONT(weight_concordance, 0.75) AS p75,
        QUANTILE_CONT(weight_concordance, 0.90) AS p90,
        QUANTILE_CONT(weight_concordance, 0.95) AS p95,
        QUANTILE_CONT(weight_concordance, 0.99) AS p99,

        COUNT(*) FILTER (
            WHERE weight_concordance >= 0.50
        ) AS n_ge_050,

        COUNT(*) FILTER (
            WHERE weight_concordance >= 0.75
        ) AS n_ge_075,

        COUNT(*) FILTER (
            WHERE weight_concordance >= 0.90
        ) AS n_ge_090,

        COUNT(*) FILTER (
            WHERE weight_concordance >= 0.95
        ) AS n_ge_095

    FROM read_parquet('{mirror_path}')

    WHERE has_both_mirrors
      AND export_value_fob > 0
      AND import_value_cif > 0
      AND export_weight_kg > 0
      AND import_weight_kg > 0
    """
).df()

weight_concordance_summary

,n_comparable_groups,p01,p05,p10,p25,median,p75,p90,p95,p99,n_ge_050,n_ge_075,n_ge_090,n_ge_095
0,2303626,0.00015,0.003179,0.013398,0.093155,0.373687,0.742253,0.935992,0.982776,1.0,951439,563570,304846,199142


In [23]:
eligible_factor_summary = duckdb.sql(
    f"""
    SELECT
        COUNT(*) AS n_eligible,

        MEDIAN(raw_value_ratio)
            AS median_raw_ratio,

        MEDIAN(unit_value_ratio)
            AS median_unit_value_ratio,

        QUANTILE_CONT(
            unit_value_ratio,
            0.01
        ) AS unit_ratio_p01,

        QUANTILE_CONT(
            unit_value_ratio,
            0.05
        ) AS unit_ratio_p05,

        QUANTILE_CONT(
            unit_value_ratio,
            0.95
        ) AS unit_ratio_p95,

        QUANTILE_CONT(
            unit_value_ratio,
            0.99
        ) AS unit_ratio_p99

    FROM read_parquet('{mirror_path}')

    WHERE training_eligible
    """
).df()

eligible_factor_summary

,n_eligible,median_raw_ratio,median_unit_value_ratio,unit_ratio_p01,unit_ratio_p05,unit_ratio_p95,unit_ratio_p99
0,304846,1.047952,1.048396,0.159362,0.524605,2.125901,7.444549


In [24]:
factor_source_summary = duckdb.sql(
    f"""
    WITH direct AS (
        SELECT
            'direct' AS factor_source,
            import_direct_cif_fob_factor AS factor
        FROM read_parquet('{mirror_path}')
        WHERE import_direct_cif_fob_factor > 0
    ),

    mirror AS (
        SELECT
            'mirror' AS factor_source,
            unit_value_ratio AS factor
        FROM read_parquet('{mirror_path}')
        WHERE training_eligible
          AND unit_value_ratio > 0
    ),

    combined AS (
        SELECT * FROM direct
        UNION ALL
        SELECT * FROM mirror
    )

    SELECT
        factor_source,
        COUNT(*) AS n,

        QUANTILE_CONT(factor, 0.001) AS p001,
        QUANTILE_CONT(factor, 0.01) AS p01,
        QUANTILE_CONT(factor, 0.05) AS p05,
        MEDIAN(factor) AS median,
        QUANTILE_CONT(factor, 0.95) AS p95,
        QUANTILE_CONT(factor, 0.99) AS p99,
        QUANTILE_CONT(factor, 0.999) AS p999,

        COUNT(*) FILTER (
            WHERE factor < 1
        ) AS n_below_1,

        COUNT(*) FILTER (
            WHERE factor BETWEEN 1 AND 1.5
        ) AS n_1_to_1_5,

        COUNT(*) FILTER (
            WHERE factor > 1.5
        ) AS n_above_1_5

    FROM combined
    GROUP BY factor_source
    ORDER BY factor_source
    """
).df()

factor_source_summary

,factor_source,n,p001,p01,p05,median,p95,p99,p999,n_below_1,n_1_to_1_5,n_above_1_5
0,direct,683328,0.997603,1.000000,1.000000,1.074112,1.729669,6.666726,83.501982,3900,630909,48519
1,mirror,304846,0.020420,0.159362,0.524605,1.048396,2.125901,7.444549,86.110448,97820,174995,32031


In [25]:
factor_overlap_summary = duckdb.sql(
    f"""
    SELECT
        COUNT(*) AS n_mirror_groups,

        COUNT(*) FILTER (
            WHERE import_direct_cif_fob_factor > 0
        ) AS n_with_direct,

        COUNT(*) FILTER (
            WHERE training_eligible
        ) AS n_with_eligible_mirror,

        COUNT(*) FILTER (
            WHERE import_direct_cif_fob_factor > 0
              AND training_eligible
        ) AS n_with_both,

        MEDIAN(
            unit_value_ratio
            / NULLIF(
                import_direct_cif_fob_factor,
                0
            )
        ) FILTER (
            WHERE import_direct_cif_fob_factor > 0
              AND training_eligible
              AND unit_value_ratio > 0
        ) AS median_mirror_to_direct_ratio,

        QUANTILE_CONT(
            unit_value_ratio
            / NULLIF(
                import_direct_cif_fob_factor,
                0
            ),
            0.05
        ) FILTER (
            WHERE import_direct_cif_fob_factor > 0
              AND training_eligible
              AND unit_value_ratio > 0
        ) AS p05_mirror_to_direct_ratio,

        QUANTILE_CONT(
            unit_value_ratio
            / NULLIF(
                import_direct_cif_fob_factor,
                0
            ),
            0.95
        ) FILTER (
            WHERE import_direct_cif_fob_factor > 0
              AND training_eligible
              AND unit_value_ratio > 0
        ) AS p95_mirror_to_direct_ratio

    FROM read_parquet('{mirror_path}')
    """
).df()

factor_overlap_summary

,n_mirror_groups,n_with_direct,n_with_eligible_mirror,n_with_both,median_mirror_to_direct_ratio,p05_mirror_to_direct_ratio,p95_mirror_to_direct_ratio
0,5487981,683328,304846,51899,0.999787,0.470065,1.755435


In [26]:
import duckdb

training_path = (
    "/storage/vacoda/data/interim/cif_fob/target_h5/"
    "target_h5_cif_fob_training_sample.parquet"
)

candidate_summary = duckdb.sql(
    f"""
    SELECT
        factor_source,
        factor_quality_band,
        pair_uses_estimated_weight,
        pair_uses_weight_proxy,

        COUNT(*) AS n,

        MEDIAN(observed_cif_fob_factor) AS median_factor,

        QUANTILE_CONT(
            observed_cif_fob_factor,
            0.05
        ) AS p05,

        QUANTILE_CONT(
            observed_cif_fob_factor,
            0.95
        ) AS p95,

        MEDIAN(
            CASE
                WHEN factor_source = 'direct'
                    THEN import_direct_covered_primary_value
                ELSE import_value_cif
            END
        ) AS median_evidence_value

    FROM read_parquet('{training_path}')

    WHERE model_eligible

    GROUP BY
        factor_source,
        factor_quality_band,
        pair_uses_estimated_weight,
        pair_uses_weight_proxy

    ORDER BY
        factor_source,
        factor_quality_band,
        pair_uses_estimated_weight,
        pair_uses_weight_proxy
    """
).df()

candidate_summary

,factor_source,factor_quality_band,pair_uses_estimated_weight,pair_uses_weight_proxy,n,median_factor,p05,p95,median_evidence_value
0,direct,below_one,False,False,3334,0.999476,0.917431,0.999793,1797.6115
1,direct,below_one,False,True,1,0.997658,0.997658,0.997658,80.9520
2,direct,below_one,True,False,564,0.999474,0.984884,0.999764,10740.8730
3,direct,below_one,True,True,1,0.999003,0.999003,0.999003,55500.6050
4,direct,extreme_above_3,False,False,13900,6.000010,3.176657,71.056874,326.0750
5,direct,extreme_above_3,False,True,35,5.811426,3.090695,35.500000,105.7700
6,direct,extreme_above_3,True,False,1413,5.058824,3.091492,49.332906,396.3700
7,direct,extreme_above_3,True,True,2,3.124265,3.116103,3.132426,659.0000
8,direct,high_1_5_to_3,False,False,26568,1.801314,1.519746,2.719740,1005.6700
9,direct,high_1_5_to_3,False,True,216,1.709344,1.515817,2.500000,1414.6155


In [27]:
value_threshold_audit = duckdb.sql(
    f"""
    WITH candidates AS (
        SELECT
            factor_source,
            observed_cif_fob_factor,

            CASE
                WHEN factor_source = 'direct'
                    THEN import_direct_covered_primary_value

                WHEN factor_source = 'mirror'
                    THEN LEAST(
                        export_value_fob,
                        import_value_cif
                    )
            END AS evidence_value

        FROM read_parquet('{training_path}')

        WHERE model_eligible
    ),

    thresholds AS (
        SELECT * FROM (
            VALUES
                (0),
                (100),
                (500),
                (1000),
                (5000),
                (10000)
        ) AS t(minimum_value)
    )

    SELECT
        c.factor_source,
        t.minimum_value,

        COUNT(*) FILTER (
            WHERE c.evidence_value >= t.minimum_value
        ) AS n_retained,

        MEDIAN(c.observed_cif_fob_factor)
            FILTER (
                WHERE c.evidence_value >= t.minimum_value
            ) AS median_factor,

        QUANTILE_CONT(
            c.observed_cif_fob_factor,
            0.05
        ) FILTER (
            WHERE c.evidence_value >= t.minimum_value
        ) AS p05,

        QUANTILE_CONT(
            c.observed_cif_fob_factor,
            0.95
        ) FILTER (
            WHERE c.evidence_value >= t.minimum_value
        ) AS p95,

        COUNT(*) FILTER (
            WHERE c.evidence_value >= t.minimum_value
              AND c.observed_cif_fob_factor < 1
        ) AS n_below_one,

        COUNT(*) FILTER (
            WHERE c.evidence_value >= t.minimum_value
              AND c.observed_cif_fob_factor > 3
        ) AS n_above_three

    FROM candidates AS c
    CROSS JOIN thresholds AS t

    GROUP BY
        c.factor_source,
        t.minimum_value

    ORDER BY
        c.factor_source,
        t.minimum_value
    """
).df()

value_threshold_audit

,factor_source,minimum_value,n_retained,median_factor,p05,p95,n_below_one,n_above_three
0,direct,0,682692,1.074108,1.000000,1.723187,3900,15350
1,direct,100,610413,1.073967,1.000000,1.616993,2865,10857
2,direct,500,537717,1.072963,1.000000,1.500000,2442,6625
3,direct,1000,499192,1.072529,1.000000,1.454957,2239,5012
4,direct,5000,393816,1.070995,1.000000,1.359781,1697,2342
5,direct,10000,344849,1.069908,1.000000,1.328317,1439,1703
6,mirror,0,252995,1.044253,0.515999,2.133494,84152,7440
7,mirror,100,245162,1.044061,0.533724,1.994887,81448,5784
8,mirror,500,234856,1.043833,0.553665,1.887669,77858,4492
9,mirror,1000,228144,1.043880,0.567422,1.844988,75391,3938


In [ ]:
examples = {}

for _, example_row in example_keys.iterrows():
    structure_class = example_row["structure_class"]

    examples[structure_class] = inspect_group(
        example_row
    )

    print("=" * 100)
    print(structure_class)
    print(example_row.to_dict())
    display(examples[structure_class])

partner2_detail_x_mot_detail
{'structure_class': 'partner2_detail_x_mot_detail', 'year': 2025, 'reporter_iso': 'LVA', 'partner_iso': 'LTU', 'flow_code': 'X', 'target_cmd': '440910', 'n_reported_rows': 3}


,hs_sync_input_row_id,year,reporter_iso,partner_iso,flow_code,target_cmd,partner2_iso,partner2_name,mot_code,mot_name,...,source_cmd_original,primary_value_original,primary_value_hs_synced,allocation_weight,cif_value_usd,fob_value_usd,canonical_weight_hs_synced,canonical_weight_source,is_reported,is_aggregate
0,5476794,2025,LVA,LTU,X,440910,_X,"Areas, nes",3200,Road,...,440910,593253.573,593253.573,1.0,NaN,593253.573,499244.0,reported_net_weight_kg,True,False
1,5476783,2025,LVA,LTU,X,440910,_X,"Areas, nes",9200,"Postal consignments, mail or courier shipment",...,440910,38.419,38.419,1.0,NaN,38.419,33.0,reported_net_weight_kg,True,False
2,5476786,2025,LVA,LTU,X,440910,_X,"Areas, nes",9900,Other,...,440910,55537.541,55537.541,1.0,NaN,55537.541,49879.0,reported_net_weight_kg,True,False


partner2_detail_x_mot_total
{'structure_class': 'partner2_detail_x_mot_total', 'year': 2025, 'reporter_iso': 'FIN', 'partner_iso': 'FRA', 'flow_code': 'X', 'target_cmd': '481013', 'n_reported_rows': 3}


,hs_sync_input_row_id,year,reporter_iso,partner_iso,flow_code,target_cmd,partner2_iso,partner2_name,mot_code,mot_name,...,source_cmd_original,primary_value_original,primary_value_hs_synced,allocation_weight,cif_value_usd,fob_value_usd,canonical_weight_hs_synced,canonical_weight_source,is_reported,is_aggregate
0,20828283,2025,FIN,FRA,X,481013,AUT,Austria,0,TOTAL MOT,...,481013,26973.365,26973.365,1.0,NaN,26973.365,16277.0,reported_net_weight_kg,True,False
1,20828284,2025,FIN,FRA,X,481013,DEU,Germany,0,TOTAL MOT,...,481013,3865.047,3865.047,1.0,NaN,3865.047,8221.0,reported_net_weight_kg,True,False
2,20828437,2025,FIN,FRA,X,481013,_X,"Areas, nes",0,TOTAL MOT,...,481013,4347369.476,4347369.476,1.0,NaN,4347369.476,2230704.0,reported_net_weight_kg,True,False


partner2_total_x_mot_detail
{'structure_class': 'partner2_total_x_mot_detail', 'year': 2025, 'reporter_iso': 'MYS', 'partner_iso': 'PHL', 'flow_code': 'X', 'target_cmd': '482190', 'n_reported_rows': 3}


,hs_sync_input_row_id,year,reporter_iso,partner_iso,flow_code,target_cmd,partner2_iso,partner2_name,mot_code,mot_name,...,source_cmd_original,primary_value_original,primary_value_hs_synced,allocation_weight,cif_value_usd,fob_value_usd,canonical_weight_hs_synced,canonical_weight_source,is_reported,is_aggregate
0,33547301,2025,MYS,PHL,X,482190,W00,World,1000,Air,...,482190,81358.890,81358.890,1.0,NaN,81358.890,731593.742,reported_net_weight_kg,True,False
1,33547369,2025,MYS,PHL,X,482190,W00,World,2000,Water,...,482190,37905.443,37905.443,1.0,NaN,37905.443,4483.565,reported_net_weight_kg,True,False
2,33547231,2025,MYS,PHL,X,482190,W00,World,3200,Road,...,482190,11711.259,11711.259,1.0,NaN,11711.259,343.800,reported_net_weight_kg,True,False


partner2_total_x_mot_total
{'structure_class': 'partner2_total_x_mot_total', 'year': 2025, 'reporter_iso': 'ITA', 'partner_iso': 'HUN', 'flow_code': 'X', 'target_cmd': '441299', 'n_reported_rows': 3}


,hs_sync_input_row_id,year,reporter_iso,partner_iso,flow_code,target_cmd,partner2_iso,partner2_name,mot_code,mot_name,...,source_cmd_original,primary_value_original,primary_value_hs_synced,allocation_weight,cif_value_usd,fob_value_usd,canonical_weight_hs_synced,canonical_weight_source,is_reported,is_aggregate
0,7776821,2025,ITA,HUN,X,441299,W00,World,0,TOTAL MOT,...,441291,1640.736,1640.736,1.0,NaN,1640.736,1555.0,reported_net_weight_kg,True,False
1,7806346,2025,ITA,HUN,X,441299,W00,World,0,TOTAL MOT,...,441292,41828.586,41828.586,1.0,NaN,41828.586,11729.0,reported_net_weight_kg,True,False
2,8035784,2025,ITA,HUN,X,441299,W00,World,0,TOTAL MOT,...,441299,83284.277,83284.277,1.0,NaN,83284.277,18767.0,reported_net_weight_kg,True,False


single_row
{'structure_class': 'single_row', 'year': 2025, 'reporter_iso': 'AUS', 'partner_iso': 'PAK', 'flow_code': 'M', 'target_cmd': '482010', 'n_reported_rows': 1}


,hs_sync_input_row_id,year,reporter_iso,partner_iso,flow_code,target_cmd,partner2_iso,partner2_name,mot_code,mot_name,...,source_cmd_original,primary_value_original,primary_value_hs_synced,allocation_weight,cif_value_usd,fob_value_usd,canonical_weight_hs_synced,canonical_weight_source,is_reported,is_aggregate
0,30790772,2025,AUS,PAK,M,482010,W00,World,2100,Sea,...,482010,2563.85,2563.85,1.0,2563.85,2401.24,936.288,estimated_quantity_kg,True,False


## Additional

In [6]:
import pandas as pd

path = "/home/saxena/Projects/vacoda/references/geography/source/dist_cepii.xls"

xls = pd.ExcelFile(path)

print(xls.sheet_names)

['dist_cepii']


In [27]:
import duckdb

pair_path = (
    "/home/saxena/Projects/vacoda/references/geography/prepared/country_pair_geography.parquet"
)

country_path = (
    "/home/saxena/Projects/vacoda/references/geography/prepared/country_geography.parquet"
)


duckdb.sql(
    f"""
    WITH pair_countries AS (
        SELECT origin_iso3 AS country_iso3
        FROM read_parquet('{pair_path}')

        UNION

        SELECT destination_iso3 AS country_iso3
        FROM read_parquet('{pair_path}')
    )

    SELECT
        c.country_iso3

    FROM read_parquet('{country_path}') AS c

    LEFT JOIN pair_countries AS p
      USING (country_iso3)

    WHERE p.country_iso3 IS NULL

    ORDER BY c.country_iso3
    """
).df()

,country_iso3
0,ATF


In [28]:
training_path = (
    "/storage/vacoda/data/interim/cif_fob/target_h5/"
    "target_h5_cif_fob_training_sample.parquet"
)

coverage = duckdb.sql(
    f"""
    WITH trade_countries AS (
        SELECT exporter_iso AS country_iso3
        FROM read_parquet('{training_path}')

        UNION

        SELECT importer_iso AS country_iso3
        FROM read_parquet('{training_path}')
    ),

    geography_countries AS (
        SELECT country_iso3
        FROM read_parquet('{country_path}')
    ),

    pair_countries AS (
        SELECT origin_iso3 AS country_iso3
        FROM read_parquet('{pair_path}')

        UNION

        SELECT destination_iso3 AS country_iso3
        FROM read_parquet('{pair_path}')
    )

    SELECT
        t.country_iso3,

        g.country_iso3 IS NOT NULL
            AS found_in_country_geography,

        p.country_iso3 IS NOT NULL
            AS found_in_pair_geography

    FROM trade_countries AS t

    LEFT JOIN geography_countries AS g
      USING (country_iso3)

    LEFT JOIN pair_countries AS p
      USING (country_iso3)

    WHERE g.country_iso3 IS NULL
       OR p.country_iso3 IS NULL

    ORDER BY t.country_iso3
    """
).df()

coverage

,country_iso3,found_in_country_geography,found_in_pair_geography
0,A59,False,False
1,A79,False,False
2,ASM,False,False
3,ATA,False,False
4,ATF,True,False
5,BES,False,False
6,BLM,False,False
7,BVT,False,False
8,COD,False,False
9,CUW,False,False


In [30]:
import pandas as pd

dist_path = (
    "/home/saxena/Projects/vacoda/references/geography/source/dist_cepii.xls"
)

raw = pd.read_excel(
    dist_path,
    sheet_name="dist_cepii",
)

distance_cols = [
    "dist",
    "distcap",
    "distw",
    "distwces",
]

distance_numeric = raw[distance_cols].apply(
    pd.to_numeric,
    errors="coerce",
)

pd.DataFrame({
    "available": distance_numeric.notna().sum(),
    "missing": distance_numeric.isna().sum(),
})

,available,missing
dist,50176,0
distcap,50176,0
distw,47961,2215
distwces,47961,2215


In [31]:
missing_distw = distance_numeric["distw"].isna()

pd.Series({
    "distw_missing": missing_distw.sum(),

    "recovered_by_distwces": (
        missing_distw
        & distance_numeric["distwces"].notna()
    ).sum(),

    "recovered_by_distcap_after_distwces": (
        missing_distw
        & distance_numeric["distwces"].isna()
        & distance_numeric["distcap"].notna()
    ).sum(),

    "recovered_by_dist_after_others": (
        missing_distw
        & distance_numeric["distwces"].isna()
        & distance_numeric["distcap"].isna()
        & distance_numeric["dist"].notna()
    ).sum(),

    "still_missing_all_measures": (
        distance_numeric[
            ["distw", "distwces", "distcap", "dist"]
        ]
        .isna()
        .all(axis=1)
    ).sum(),
})

distw_missing                          2215
recovered_by_distwces                     0
recovered_by_distcap_after_distwces    2215
recovered_by_dist_after_others            0
still_missing_all_measures                0
dtype: int64

## Why traiing featuees has incomplete candidates

In [32]:
import duckdb

features_path = (
    "/storage/vacoda/data/interim/cif_fob/target_h5/"
    "target_h5_cif_fob_training_features.parquet"
)

candidate_missing_reasons = duckdb.sql(
    f"""
    SELECT
        predictor_completeness_reason,
        COUNT(*) AS n_candidates,
        ROUND(
            100.0 * COUNT(*)
            / SUM(COUNT(*)) OVER (),
            2
        ) AS share_incomplete_percent

    FROM read_parquet('{features_path}')

    WHERE model_eligible
      AND NOT initial_predictors_complete

    GROUP BY predictor_completeness_reason

    ORDER BY n_candidates DESC
    """
).df()

candidate_missing_reasons

,predictor_completeness_reason,n_candidates,share_incomplete_percent
0,missing_pair_geography,107748,100.0


In [33]:
missing_country_summary = duckdb.sql(
    f"""
    WITH missing AS (
        SELECT
            exporter_iso,
            importer_iso,
            model_eligible,
            pair_geography_found,
            exporter_geography_found,
            importer_geography_found

        FROM read_parquet('{features_path}')

        WHERE model_eligible
          AND NOT initial_predictors_complete
    ),

    countries AS (
        SELECT
            exporter_iso AS country_iso3,
            'exporter' AS role,
            COUNT(*) AS n_rows

        FROM missing

        WHERE NOT exporter_geography_found
           OR NOT pair_geography_found

        GROUP BY exporter_iso

        UNION ALL

        SELECT
            importer_iso AS country_iso3,
            'importer' AS role,
            COUNT(*) AS n_rows

        FROM missing

        WHERE NOT importer_geography_found
           OR NOT pair_geography_found

        GROUP BY importer_iso
    )

    SELECT
        country_iso3,
        role,
        n_rows

    FROM countries

    ORDER BY n_rows DESC
    """
).df()

missing_country_summary.head(50)

,country_iso3,role,n_rows
0,SRB,importer,38735
1,ROU,importer,22104
2,S19,exporter,12845
3,ROU,exporter,10185
4,SRB,exporter,9058
5,E19,exporter,5399
6,SVK,importer,5343
7,_X,exporter,5015
8,CZE,importer,4342
9,LUX,importer,3715


In [34]:
predictor_summary = duckdb.sql(
    f"""
    SELECT
        MIN(distance_model_km) AS min_distance,
        MEDIAN(distance_model_km) AS median_distance,
        MAX(distance_model_km) AS max_distance,

        MIN(log_distance) AS min_log_distance,
        MAX(log_distance) AS max_log_distance,

        MIN(crude_oil_average_usd_per_bbl) AS min_oil_price,
        MAX(crude_oil_average_usd_per_bbl) AS max_oil_price,

        MIN(product_median_unit_value) AS min_product_uv,
        MEDIAN(product_median_unit_value) AS median_product_uv,
        QUANTILE_CONT(
            product_median_unit_value,
            0.99
        ) AS product_uv_p99,
        MAX(product_median_unit_value) AS max_product_uv,

        MIN(product_weight_to_value_ratio) AS min_weight_value_ratio,
        MAX(product_weight_to_value_ratio) AS max_weight_value_ratio

    FROM read_parquet('{features_path}')

    WHERE initial_predictors_complete
    """
).df()

predictor_summary

,min_distance,median_distance,max_distance,min_log_distance,max_log_distance,min_oil_price,max_oil_price,min_product_uv,median_product_uv,product_uv_p99,max_product_uv,min_weight_value_ratio,max_weight_value_ratio
0,60.77057,4385.531,19737.02,4.107106,9.890251,41.26,97.1,0.084825,4.066314,60.08792,109.703862,0.009115,11.788927


In [35]:
product_source_by_year = duckdb.sql(
    f"""
    SELECT
        year,
        product_median_unit_value_source,
        COUNT(*) AS n_rows

    FROM read_parquet('{features_path}')

    GROUP BY
        year,
        product_median_unit_value_source

    ORDER BY
        year,
        product_median_unit_value_source
    """
).df()

product_source_by_year

,year,product_median_unit_value_source,n_rows
0,2017,global_year,13
1,2017,product_group_year,665883
2,2018,global_year,10
3,2018,lagged_product,657282
4,2018,product_group_year,25
5,2019,global_year,4
6,2019,lagged_product,655076
7,2019,product_group_year,26
8,2020,lagged_product,617531
9,2020,product_group_year,61


In [36]:
coverage_by_factor_source = duckdb.sql(
    f"""
    SELECT
        factor_source,

        COUNT(*) AS n_model_candidates,

        COUNT(*) FILTER (
            WHERE initial_predictors_complete
        ) AS n_complete,

        ROUND(
            100.0
            * COUNT(*) FILTER (
                WHERE initial_predictors_complete
            )
            / COUNT(*),
            2
        ) AS complete_percent

    FROM read_parquet('{features_path}')

    WHERE model_eligible

    GROUP BY factor_source

    ORDER BY factor_source
    """
).df()

coverage_by_factor_source

,factor_source,n_model_candidates,n_complete,complete_percent
0,direct,682692,595745,87.26
1,mirror,252995,232194,91.78


In [37]:
import duckdb

country_path = (
    "/home/saxena/Projects/vacoda/"
    "references/geography/prepared/"
    "country_geography.parquet"
)

pair_path = (
    "/home/saxena/Projects/vacoda/"
    "references/geography/prepared/"
    "country_pair_geography.parquet"
)

legacy_codes = [
    "ROM",
    "ZAR",
    "TMP",
    "SCG",
    "YUG",
    "WBG",
    "SDN",
]

country_matches = duckdb.sql(
    f"""
    SELECT *
    FROM read_parquet('{country_path}')
    WHERE country_iso3 IN (
        {", ".join(repr(code) for code in legacy_codes)}
    )
    ORDER BY country_iso3
    """
).df()

pair_matches = duckdb.sql(
    f"""
    SELECT DISTINCT origin_iso3 AS country_iso3
    FROM read_parquet('{pair_path}')
    WHERE origin_iso3 IN (
        {", ".join(repr(code) for code in legacy_codes)}
    )

    UNION

    SELECT DISTINCT destination_iso3 AS country_iso3
    FROM read_parquet('{pair_path}')
    WHERE destination_iso3 IN (
        {", ".join(repr(code) for code in legacy_codes)}
    )

    ORDER BY country_iso3
    """
).df()

country_matches
pair_matches

,country_iso3
0,ROM
1,SDN
2,TMP
3,YUG
4,ZAR


In [38]:
features_path = (
    "/storage/vacoda/data/interim/cif_fob/target_h5/"
    "target_h5_cif_fob_training_features.parquet"
)

missing_value_summary = duckdb.sql(
    f"""
    SELECT
        CASE
            WHEN exporter_iso IN ('SRB', 'ROU', 'MNE')
                THEN exporter_iso

            WHEN importer_iso IN ('SRB', 'ROU', 'MNE')
                THEN importer_iso

            WHEN exporter_iso IN ('S19', 'E19', '_X')
                THEN exporter_iso

            WHEN importer_iso IN ('S19', 'E19', '_X')
                THEN importer_iso

            ELSE 'other'
        END AS missing_entity,

        COUNT(*) AS n_candidates,

        SUM(
            CASE
                WHEN factor_source = 'direct'
                    THEN import_direct_covered_primary_value

                WHEN factor_source = 'mirror'
                    THEN LEAST(
                        export_value_fob,
                        import_value_cif
                    )
            END
        ) AS evidence_value,

        MEDIAN(observed_cif_fob_factor)
            AS median_factor

    FROM read_parquet('{features_path}')

    WHERE model_eligible
      AND NOT initial_predictors_complete

    GROUP BY 1

    ORDER BY evidence_value DESC NULLS LAST
    """
).df()

missing_value_summary

,missing_entity,n_candidates,evidence_value,median_factor
0,ROU,31717,3.006769e+10,1.042000
1,S19,14822,1.885266e+10,1.070626
2,SRB,46385,1.875951e+10,1.012821
3,E19,5355,5.125772e+09,1.000000
4,other,6856,2.022069e+09,1.045999
5,MNE,2613,3.278552e+08,1.025799


In [39]:
import duckdb

features_path = (
    "/storage/vacoda/data/interim/cif_fob/target_h5/"
    "target_h5_cif_fob_training_features.parquet"
)

other_missing = duckdb.sql(
    f"""
    WITH missing AS (
        SELECT
            exporter_iso,
            importer_iso,

            CASE
                WHEN factor_source = 'direct'
                    THEN import_direct_covered_primary_value

                WHEN factor_source = 'mirror'
                    THEN LEAST(
                        export_value_fob,
                        import_value_cif
                    )
            END AS evidence_value

        FROM read_parquet('{features_path}')

        WHERE model_eligible
          AND NOT initial_predictors_complete
    ),

    exploded AS (
        SELECT
            exporter_iso AS country_iso3,
            'exporter' AS role,
            evidence_value
        FROM missing

        UNION ALL

        SELECT
            importer_iso AS country_iso3,
            'importer' AS role,
            evidence_value
        FROM missing
    )

    SELECT
        country_iso3,
        role,
        COUNT(*) AS n_candidates,
        SUM(evidence_value) AS evidence_value

    FROM exploded

    WHERE country_iso3 NOT IN (
        'ROU',
        'SRB',
        'MNE',
        'S19',
        'E19',
        '_X'
    )

    GROUP BY
        country_iso3,
        role

    ORDER BY
        evidence_value DESC NULLS LAST,
        n_candidates DESC
    """
).df()

other_missing.head(50)

,country_iso3,role,n_candidates,evidence_value
0,SVK,importer,5343,5.611736e+09
1,USA,importer,1566,5.353764e+09
2,DEU,exporter,3130,3.628483e+09
3,CHN,exporter,2752,3.296043e+09
4,CZE,importer,4342,3.053864e+09
5,POL,exporter,2332,2.967337e+09
6,CHN,importer,276,2.454925e+09
7,ITA,exporter,2978,2.405292e+09
8,AUS,importer,1511,2.379610e+09
9,ITA,importer,590,2.211054e+09


In [40]:
import duckdb

features_path = (
    "/storage/vacoda/data/interim/cif_fob/target_h5/"
    "target_h5_cif_fob_training_features.parquet"
)

actual_missing_countries = duckdb.sql(
    f"""
    WITH missing_sides AS (
        SELECT
            exporter_iso AS country_iso3,
            'exporter' AS role,

            CASE
                WHEN factor_source = 'direct'
                    THEN import_direct_covered_primary_value

                WHEN factor_source = 'mirror'
                    THEN LEAST(
                        export_value_fob,
                        import_value_cif
                    )
            END AS evidence_value

        FROM read_parquet('{features_path}')

        WHERE model_eligible
          AND NOT exporter_geography_found

        UNION ALL

        SELECT
            importer_iso AS country_iso3,
            'importer' AS role,

            CASE
                WHEN factor_source = 'direct'
                    THEN import_direct_covered_primary_value

                WHEN factor_source = 'mirror'
                    THEN LEAST(
                        export_value_fob,
                        import_value_cif
                    )
            END AS evidence_value

        FROM read_parquet('{features_path}')

        WHERE model_eligible
          AND NOT importer_geography_found
    )

    SELECT
        country_iso3,
        role,
        COUNT(*) AS n_candidates,
        SUM(evidence_value) AS evidence_value

    FROM missing_sides

    GROUP BY
        country_iso3,
        role

    ORDER BY
        evidence_value DESC NULLS LAST,
        n_candidates DESC
    """
).df()

actual_missing_countries

,country_iso3,role,n_candidates,evidence_value
0,ROU,importer,22104,1.752895e+10
1,ROU,exporter,10185,1.310506e+10
2,SRB,importer,38735,1.213709e+10
3,S19,exporter,12845,9.538274e+09
4,S19,importer,2432,9.414495e+09
5,SRB,exporter,9058,7.433095e+09
6,E19,exporter,5399,5.126274e+09
7,_X,exporter,5015,1.822415e+09
8,MNE,importer,1937,3.897548e+08
9,MNE,exporter,1015,1.417651e+08


In [41]:
missing_pairs = duckdb.sql(
    f"""
    SELECT
        exporter_iso,
        importer_iso,
        COUNT(*) AS n_candidates,

        SUM(
            CASE
                WHEN factor_source = 'direct'
                    THEN import_direct_covered_primary_value

                WHEN factor_source = 'mirror'
                    THEN LEAST(
                        export_value_fob,
                        import_value_cif
                    )
            END
        ) AS evidence_value,

        BOOL_AND(exporter_geography_found)
            AS exporter_country_found,

        BOOL_AND(importer_geography_found)
            AS importer_country_found

    FROM read_parquet('{features_path}')

    WHERE model_eligible
      AND NOT pair_geography_found

    GROUP BY
        exporter_iso,
        importer_iso

    ORDER BY
        evidence_value DESC NULLS LAST,
        n_candidates DESC
    """
).df()

missing_pairs.head(100)

,exporter_iso,importer_iso,n_candidates,evidence_value,exporter_country_found,importer_country_found
0,E19,SVK,2086,4.155906e+09,False,True
1,S19,USA,844,4.013268e+09,False,True
2,POL,ROU,903,2.246520e+09,True,False
3,DEU,ROU,1011,2.124203e+09,True,False
4,CHN,S19,241,1.904260e+09,True,False
...,...,...,...,...,...,...
95,SRB,CHN,39,1.746842e+08,False,True
96,S19,TUR,541,1.742654e+08,False,True
97,SRB,AUT,204,1.684196e+08,False,True
98,ROU,ESP,78,1.591703e+08,False,True


In [42]:
pair_path = (
    "/home/saxena/Projects/vacoda/"
    "references/geography/prepared/"
    "country_pair_geography.parquet"
)

missing_pair_entities = duckdb.sql(
    f"""
    WITH pair_universe AS (
        SELECT origin_iso3 AS country_iso3
        FROM read_parquet('{pair_path}')

        UNION

        SELECT destination_iso3 AS country_iso3
        FROM read_parquet('{pair_path}')
    ),

    missing_routes AS (
        SELECT DISTINCT
            exporter_iso,
            importer_iso

        FROM read_parquet('{features_path}')

        WHERE model_eligible
          AND NOT pair_geography_found
    )

    SELECT
        country_iso3,
        role,
        COUNT(*) AS n_missing_routes

    FROM (
        SELECT
            m.exporter_iso AS country_iso3,
            'exporter' AS role

        FROM missing_routes AS m

        LEFT JOIN pair_universe AS p
          ON m.exporter_iso = p.country_iso3

        WHERE p.country_iso3 IS NULL

        UNION ALL

        SELECT
            m.importer_iso AS country_iso3,
            'importer' AS role

        FROM missing_routes AS m

        LEFT JOIN pair_universe AS p
          ON m.importer_iso = p.country_iso3

        WHERE p.country_iso3 IS NULL
    )

    GROUP BY
        country_iso3,
        role

    ORDER BY
        n_missing_routes DESC
    """
).df()

missing_pair_entities

,country_iso3,role,n_missing_routes
0,SRB,importer,158
1,ROU,importer,141
2,S19,exporter,136
3,ROU,exporter,125
4,SRB,exporter,109
5,S19,importer,96
6,MNE,importer,49
7,COD,exporter,42
8,MNE,exporter,41
9,COD,importer,36
